In [1]:
%display latex

Unrecognized magic `%display`.

Julia does not use the IPython `%magic` syntax.   To interact with the IJulia kernel, use `IJulia.somefunction(...)`, for example.  Julia macros, string macros, and functions can be used to accomplish most of the other functionalities of IPython magics.


# PF Spin-s  

We explore the WKB periods for the spin-$$s$$ spheroidal wave action density. 
The hamiltonian for the system is
$$ H(q,p) = \frac{p^2}{2} + \sin^2 q + \frac{\mu_0+\mu_1 \sin q}{\cos^2 q} + \mu_2 \sin q $$
and treat the tilt $\mu_2 = i \sqrt{2} \hbar s $ as a perturbative ``quantum'' correction.

This notebook in particular computes:
- Computes the PF system for the spin-s spheroidal WKB periods

- notebook reproduces the known slice $$\mu_1=\mu_2=0$$ after the right convention change $$E \mapsto 1-E$$ and $$ \mu_0 \mapsto -\mu_0$$

In [5]:
using Pkg
using Oscar

  ___   ___   ___    _    ____
 / _ \ / __\ / __\  / \  |  _ \  | Combining and extending ANTIC, GAP,
| |_| |\__ \| |__  / ^ \ |  ´ /  | Polymake and Singular
 \___/ \___/ \___//_/ \_\|_|\_\  | Type "?Oscar" for more information
o--------o-----o-----o--------o  | Documentation: https://docs.oscar-system.org
  S Y M B O L I C   T O O L S    | Version 1.7.2


## Graded Polynomial Ring and Jacobian Ideal

$(q,p) \mapsto (\arcsin(x), y \, \partial_x \arcsin(x))$ maps the surface $H=E$ to a quartic curve $$f=0.$$ We work in a graded polynomial ring $R$ over the rational field with weights $[1,2,1]$ corresponding to weighted projective space $[x,y,z]\in \mathbf{P}^{(1,2,1)}$ where $z=0$ describes the hyperplane at infinity.
f = 0. Here we compute the Jacobian ideal and the groebner basis for $R$ and reduce 

In [6]:
# Parameter ring and coefficient field
T, (mu0, mu1, mu2, E) = polynomial_ring(QQ, [:mu0, :mu1, :mu2, :E])
K = fraction_field(T)

# weighted ring of P(1,2,1)
R, (x, y, z) = graded_polynomial_ring(K, [:x, :y, :z], [1, 2, 1])

f = K(1//2)*y^2 +
    x^4 -
    mu2*x^3*z +
    (E - K(2))*x^2*z^2 +
    (mu1 + mu2)*x*z^3 +
    (mu0 - E + K(1))*z^4

@assert is_homogeneous(f)
@assert degree(Int, f) == 4

# Jacobian ideal
J = ideal(R, [derivative(f, x), derivative(f, y), derivative(f, z)])

# Gröbner basis; default_ordering(R) is already weighted here,
# but you can also pass ordering = wdegrevlex(R, [1,2,1]) explicitly.
G = groebner_basis(J)

reduce_mod_J(g) = normal_form(g, J)

println("f =", f)
println("df/dx =", derivative(f, x))
println("df/dy =", derivative(f, y))
println("df/dz =", derivative(f, z))

f =x^4 - mu2*x^3*z + (E - 2)*x^2*z^2 + (mu1 + mu2)*x*z^3 + 1//2*y^2 + (mu0 - E + 1)*z^4
df/dx =4*x^3 - 3*mu2*x^2*z + (2*E - 4)*x*z^2 + (mu1 + mu2)*z^3
df/dy =y
df/dz =-mu2*x^3 + (2*E - 4)*x^2*z + (3*mu1 + 3*mu2)*x*z^2 + (4*mu0 - 4*E + 4)*z^3


In [7]:
J

Ideal generated by
  4*x^3 - 3*mu2*x^2*z + (2*E - 4)*x*z^2 + (mu1 + mu2)*z^3
  y
  -mu2*x^3 + (2*E - 4)*x^2*z + (3*mu1 + 3*mu2)*x*z^2 + (4*mu0 - 4*E + 4)*z^3

##### set parameter derivatives  $\nabla_{\vec \mu}f$

In [8]:
dmu0 = z^4          # ∂_{\mu_0} f
dmu1 = x*z^3
dmu2 = -x^3*z + x*z^3
dE   = x^2*z^2 - z^4

# return the normal form of the derivs $\nabla_{\mu_i}f$ with respect to the ideal
for (name, g) in [(:mu0, dmu0), (:mu1, dmu1), (:mu2, dmu2), (:E, dE)]
    println(name, " => ", reduce_mod_J(g))
end

mu0 => z^4
mu1 => (-4*mu0*mu1 + 3//4*mu0*mu2^3 - 8//3*mu0*mu2*E + 4//3*mu0*mu2 - 1//4*mu1^2*mu2 + 1//12*mu1*mu2^2*E - 2//3*mu1*mu2^2 - 1//3*mu1*E^2 + 16//3*mu1*E - 16//3*mu1 - 2//3*mu2^3*E + 1//3*mu2^3 + 7//3*mu2*E^2 - 8//3*mu2*E)//(mu0*mu2^2 - 8//3*mu0*E + 16//3*mu0 + 3*mu1^2 - 1//2*mu1*mu2^3 + 7//3*mu1*mu2*E + 4//3*mu1*mu2 - 1//2*mu2^4 - 1//6*mu2^2*E^2 + 2*mu2^2*E - 4//3*mu2^2 + 2//3*E^3 - 4//3*E^2)*z^4
mu2 => (-4*mu0^2*mu2 - 1//4*mu0*mu1*mu2^2 - 8//3*mu0*mu1*E + 4//3*mu0*mu1 + 1//2*mu0*mu2^3 - 1//3*mu0*mu2*E^2 + 4*mu0*mu2*E - 8//3*mu0*mu2 + 3//4*mu1^3 + 1//12*mu1^2*mu2*E + 11//6*mu1^2*mu2 + 1//2*mu1*mu2^2*E + mu1*mu2^2 + 7//3*mu1*E^2 - 8//3*mu1*E - 1//3*mu2^3*E + 2//3*mu2^3 + 1//3*mu2*E^3 - 2//3*mu2*E^2)//(mu0*mu2^2 - 8//3*mu0*E + 16//3*mu0 + 3*mu1^2 - 1//2*mu1*mu2^3 + 7//3*mu1*mu2*E + 4//3*mu1*mu2 - 1//2*mu2^4 - 1//6*mu2^2*E^2 + 2*mu2^2*E - 4//3*mu2^2 + 2//3*E^3 - 4//3*E^2)*z^4
E => (16//3*mu0^2 + 2//3*mu0*mu1*mu2 + 1//2*mu0*mu2^2*E - 4//3*mu0*mu2^2 - 4//3*mu0*E^2 - 8//3*mu0*E - 1/

In [9]:
# list all possible reductions within the ideal
D = Dict(
    :mu0 => dmu0,
    :mu1 => dmu1,
    :mu2 => dmu2,
    :E   => dE,
)

for a in keys(D), b in keys(D)
    println(a, " * ", b, " => ", reduce_mod_J(D[a]*D[b]))
end

mu0 * mu0 => 0
mu0 * mu1 => 0
mu0 * mu2 => 0
mu0 * E => 0
mu1 * mu0 => 0
mu1 * mu1 => 0
mu1 * mu2 => 0
mu1 * E => 0
mu2 * mu0 => 0
mu2 * mu1 => 0
mu2 * mu2 => 0
mu2 * E => 0
E * mu0 => 0
E * mu1 => 0
E * mu2 => 0
E * E => 0


## Affine Ring and Pole Reduction

In [10]:
# Parameters
T, (mu0, mu1, mu2, E) = polynomial_ring(QQ, [:mu0, :mu1, :mu2, :E])
K = fraction_field(T)

# Affine x-ring
S, (x,) = polynomial_ring(K, [:x])

# V(x) - E
VmE = x^4 - mu2*x^3 + (E - K(2))*x^2 + (mu1 + mu2)*x + (mu0 - E + K(1))

den = K(1) - x^2

q, r = divrem(VmE, den) # compute remainder 

println("Quotient q(x) = ", q)
println("Remainder r(x) = ", r)

# Should be:
# q = 1 - E + mu2*x - x^2
# r = mu0 + mu1*x

@assert r == mu0 + mu1*x

println()
println("sigma  ~  -2*q(x) * Ω/f  - 2*r(x)/(1-x^2) * Ω/f")
println("d_mu0 sigma  ~  - Ω / ((1-x^2) f)")
println("d_mu1 sigma  ~  - x Ω / ((1-x^2) f)")
println()

# pole-free combination
polefree = -2*q
println("sigma - 2mu0*d_mu0(sigma) - 2mu1*d_mu1(sigma)  ~  ", polefree, " * Ω/f")

@assert polefree == 2*(x^2 - mu2*x + E - K(1))

Quotient q(x) = -x^2 + mu2*x - E + 1
Remainder r(x) = mu1*x + mu0

sigma  ~  -2*q(x) * Ω/f  - 2*r(x)/(1-x^2) * Ω/f
d_mu0 sigma  ~  - Ω / ((1-x^2) f)
d_mu1 sigma  ~  - x Ω / ((1-x^2) f)

sigma - 2mu0*d_mu0(sigma) - 2mu1*d_mu1(sigma)  ~  2*x^2 - 2*mu2*x + 2*E - 2 * Ω/f


In [11]:
# ------------------------------------------------------------
# Basic weighted-graded helpers
# ------------------------------------------------------------

# In some OSCAR versions, if degree(Int, g) complains,
# replace it by: Int(first(degree(g)))
wdeg(g) = degree(Int, g)

function max_hom_degree(p)
    hc = homogeneous_components(p)
    isempty(hc) && return -1
    return maximum(wdeg(c) for c in values(hc))
end


max_hom_degree (generic function with 1 method)

In [12]:
# container for normal form data
struct NFData{T}
    remainder::T
    quotients::Vector{T}
end


# Utils to generate projective quartic family P^{(1,2,1)}

In [13]:
# ------------------------------------------------------------
# weighted monomial utilities
# ------------------------------------------------------------

function weighted_degree_homogeneous(p, weights::Vector{Int})
    if iszero(p)
        return -1
    end
    ds = Int[]
    for i in 1:length(p)
        e = exponent_vector(p, i)
        push!(ds, sum(weights[j] * e[j] for j in eachindex(weights)))
    end
    d = ds[1]
    all(==(d), ds) || error("Polynomial is not homogeneous in the supplied weights.")
    return d
end

function weighted_monomials(vars::Vector, weights::Vector{Int}, d::Int)
    R = parent(vars[1])
    mons = elem_type(R)[]
    d < 0 && return mons

    function rec(i::Int, dleft::Int, acc)
        if i == length(vars)
            wi = weights[i]
            if dleft % wi == 0
                push!(mons, acc * vars[i]^(dleft ÷ wi))
            end
            return
        end

        wi = weights[i]
        for e in 0:(dleft ÷ wi)
            rec(i + 1, dleft - wi * e, acc * vars[i]^e)
        end
    end

    rec(1, d, one(R))
    return mons
end

# coordinates of p in the monomial basis "basis"
function coords_in_monomial_basis(p, basis::Vector)
    return [coeff(p, m) for m in basis]
end

# ------------------------------------------------------------
# homogeneous lift specialized to weighted polynomial rings
# ------------------------------------------------------------

"""
    lift_jacobian_homogeneous(h, J; weights=[1,2,1])

Assume h is homogeneous and h ∈ J.
Return q = [q1,...,qr] such that h = sum(q[i] * gen(J,i)).

This version works by solving a linear system in the weighted-homogeneous
monomial basis of the polynomial ring.
"""
function lift_jacobian_homogeneous(h, J; weights::Vector{Int} = [1, 2, 1])
    R  = base_ring(J)
    K  = base_ring(R)
    xs = gens(R)
    Js = gens(J)

    length(xs) == length(weights) || error("weights must match number of variables")

    D = weighted_degree_homogeneous(h, weights)
    D >= 0 || error("h must be nonzero and homogeneous")

    # weighted degree-D monomial basis of the polynomial ring
    target_basis = weighted_monomials(xs, weights, D)
    isempty(target_basis) && error("No target monomials found in weighted degree $D")

    # build unknowns block-by-block:
    # qi is a linear combination of monomials of degree D - deg(Ji)
    block_monomials = Vector{Vector{elem_type(R)}}()
    column_polys    = elem_type(R)[]

    for g in Js
        dg = weighted_degree_homogeneous(g, weights)
        mons = weighted_monomials(xs, weights, D - dg)
        push!(block_monomials, mons)
        append!(column_polys, [m * g for m in mons])
    end

    nrows = length(target_basis)
    ncols = length(column_polys)

    # A * c = b, where columns are the coefficient vectors of m*Ji
    Adata = [coeff(column_polys[j], target_basis[i]) for i in 1:nrows, j in 1:ncols]
    bvec  = [coeff(h, target_basis[i]) for i in 1:nrows]

    A = matrix(K, Adata)

    ok, sol = can_solve_with_solution(A, bvec; side = :right)
    ok || error("Could not solve homogeneous lift system; h may not lie in J.")

    # reconstruct the quotient polynomials qi
    qs = elem_type(R)[]
    idx = 1
    for mons in block_monomials
        qi = zero(R)
        for m in mons
            qi += sol[idx] * m
            idx += 1
        end
        push!(qs, qi)
    end

    # sanity check
    check = zero(R)
    for i in 1:length(Js)
        check += qs[i] * Js[i]
    end
    check == h || error("Internal error: reconstructed lift does not match h.")

    return qs
end

lift_jacobian_homogeneous

In [14]:
# ------------------------------------------------------------
# Basic weighted-graded helpers
# ------------------------------------------------------------

# In some OSCAR versions, if degree(Int, g) complains,
# replace it by: Int(first(degree(g)))
wdeg(g) = degree(Int, g)

function weighted_dim(xs)
    return sum(wdeg.(xs))
end

# get the highest degree of the homogeneous polynomial p
function max_hom_degree(p)
    hc = homogeneous_components(p)
    isempty(hc) && return -1
    return maximum(wdeg(c) for c in values(hc))
end

# ------------------------------------------------------------
# Family setup P(1,2,1) quartic
# ------------------------------------------------------------
#
# build the ring and Jacobian ideal
#
function make_family()
    T, (mu0, mu1, mu2, E) = polynomial_ring(QQ, [:mu0, :mu1, :mu2, :E])
    K = fraction_field(T)

    R, (x, y, z) = graded_polynomial_ring(K, [:x, :y, :z], [1, 2, 1])

    f = K(1//2)*y^2 +
        x^4 -
        mu2*x^3*z +
        (E - K(2))*x^2*z^2 +
        (mu1 + mu2)*x*z^3 +
        (mu0 - E + K(1))*z^4

    J = ideal(R, [derivative(f, x), derivative(f, y), derivative(f, z)])

    return (; T, K, R, x, y, z, mu0, mu1, mu2, E, f, J)
end

make_family (generic function with 1 method)

In [15]:
"""
    pole_reduce_full(p, f, J; lift_fn=nothing, weights=[1,2,1])

Fully apply Griffiths–Dwork pole reduction to the polynomial/rational
numerator `p` with respect to the weighted-homogeneous polynomial `f`
and Jacobian ideal `J`.

The function first decomposes `p` into homogeneous components and estimates
a safe upper bound on the number of pole-reduction steps needed from the
weighted degree of each component. It then repeatedly calls `pole_reduce`
up to that bound, ensuring that all reducible higher-pole terms are removed.

# Arguments
- `p`: Polynomial (or numerator expression) to be reduced.
- `f`: Weighted-homogeneous defining polynomial.
- `J`: Jacobian ideal associated with `f`.
- `lift_fn`: Optional lifting function used inside `pole_reduce`.
- `weights`: Variable weights used for weighted degree computations.

# Returns
- The fully pole-reduced form of `p`.

# Notes
- If `p` has no homogeneous components, it is returned unchanged.
- The number of iterations is determined from the weighted degree bound
  `floor((deg(c) + sum(weights)) / deg(f))` applied to each homogeneous
  component `c` of `p`.
"""
function pole_reduce_full(p, f, J; lift_fn=nothing, weights=[1,2,1])
    xs = gens(base_ring(J))
    degf = weighted_degree_homogeneous(f, weights)
    wdim = sum(weights)

    hc = homogeneous_components(p)
    isempty(hc) && return p

    max_pole_order = maximum(
        Int((weighted_degree_homogeneous(c, weights) + wdim) ÷ degf)
        for c in values(hc)
    )

    out = p
    for _ in 1:max_pole_order
        out = pole_reduce(out, f, J; lift_fn=lift_fn)
    end
    return out
end

pole_reduce_full

In [16]:
"""
    normal_form_with_quotients(p, J; lift_fn)


# Arguments
- `p`: Polynomial 
- `J`: Jacobian ideal 
- `lift_fn`: Optional lifting function used inside `pole_reduce`.

Return normal form data
"""
function normal_form_with_quotients(p, J; lift_fn=nothing)
    R = base_ring(J)
    r = normal_form(p, J)
    h = p - r

    if iszero(h)
        return NFData(r, [zero(R) for _ in 1:ngens(J)])
    end

    lift_fn === nothing && error(
        "Need a lift_fn(h, J) that returns q with h = sum(q[i]*gen(J,i))."
    )

    q = lift_fn(h, J)

    @assert length(q) == ngens(J)
    @assert h == sum(q[i] * gen(J, i) for i in 1:ngens(J))

    return NFData(r, q)
end

normal_form_with_quotients

In [17]:
fam = make_family()
lift_fn = (h, J) -> lift_jacobian_homogeneous(h, J; weights=[1,2,1])

nf = normal_form_with_quotients(fam.x^8, fam.J; lift_fn=lift_fn)

@show nf.remainder
@show nf.quotients
@assert fam.x^8 == nf.remainder + sum(nf.quotients[i] * gen(fam.J, i) for i in 1:ngens(fam.J))

nf.remainder = 0
nf.quotients = MPolyDecRingElem{AbstractAlgebra.Generic.FracFieldElem{QQMPolyRingElem}, AbstractAlgebra.Generic.MPoly{AbstractAlgebra.Generic.FracFieldElem{QQMPolyRingElem}}}[1//4*x^5 + 3//16*mu2*x^4*z + (9//64*mu2^2 - 1//8*E + 1//4)*x^3*z^2 + (-1//16*mu0^3*mu1 + 27//256*mu0^3*mu2^3 - 3//16*mu0^3*mu2*E + 5//16*mu0^3*mu2 - 11//256*mu0^2*mu1^2*mu2 + 243//4096*mu0^2*mu1*mu2^4 - 9//64*mu0^2*mu1*mu2^2*E + 25//128*mu0^2*mu1*mu2^2 + 1//32*mu0^2*mu1*E^2 + 1//16*mu0^2*mu1*E - 1//16*mu0^2*mu1 + 243//8192*mu0^2*mu2^5*E - 27//256*mu0^2*mu2^3*E^2 - 9//256*mu0^2*mu2^3*E + 17//128*mu0^2*mu2^3 + 11//128*mu0^2*mu2*E^3 + 5//64*mu0^2*mu2*E^2 - 19//32*mu0^2*mu2*E + 3//8*mu0^2*mu2 - 5//2048*mu0*mu1^3*mu2^2 - 9//256*mu0*mu1^3*E + 9//128*mu0*mu1^3 + 189//4096*mu0*mu1^2*mu2^3*E - 51//512*mu0*mu1^2*mu2^3 - 59//512*mu0*mu1^2*mu2*E^2 + 113//256*mu0*mu1^2*mu2*E - 43//128*mu0*mu1^2*mu2 + 81//4096*mu0*mu1*mu2^4*E^2 - 27//256*mu0*mu1*mu2^4*E + 3//512*mu0*mu1*mu2^4 - 45//1024*mu0*mu1*mu2^2*E^3 + 161/

# Classical Gauss Manin System 

In [18]:
"""
Combine functions above into a calcuation of the classical PF/Gauss manin 2x2 system 
"""

# ============================================================
# 0. Weighted / homogeneous utilities
# ============================================================

# Weighted degree of a homogeneous polynomial with respect to `weights`
function weighted_degree_homogeneous(p, weights::Vector{Int})
    if iszero(p)
        return -1
    end
    ds = Int[]
    for i in 1:length(p)
        e = exponent_vector(p, i)
        push!(ds, sum(weights[j] * e[j] for j in eachindex(weights)))
    end
    d = ds[1]
    all(==(d), ds) || error("Polynomial is not homogeneous in the supplied weights.")
    return d
end

# All weighted monomials of degree d in the vars xs
function weighted_monomials(xs::Vector, weights::Vector{Int}, d::Int)
    R = parent(xs[1])
    mons = elem_type(R)[]
    d < 0 && return mons

    function rec(i::Int, dleft::Int, acc)
        if i == length(xs)
            wi = weights[i]
            if dleft % wi == 0
                push!(mons, acc * xs[i]^(dleft ÷ wi))
            end
            return
        end

        wi = weights[i]
        for e in 0:(dleft ÷ wi)
            rec(i + 1, dleft - wi * e, acc * xs[i]^e)
        end
    end

    rec(1, d, one(R))
    return mons
end

# Coordinates of polynomial p in a fixed monomial basis
coords_in_monomial_basis(p, basis::Vector) = [coeff(p, m) for m in basis]

# Collect homogeneous piece of weighted degree d
function homogeneous_piece_of_degree(p, weights::Vector{Int}, d::Int)
    R = parent(p)
    out = zero(R)
    for comp in values(homogeneous_components(p))
        if !iszero(comp) && weighted_degree_homogeneous(comp, weights) == d
            out += comp
        end
    end
    return out
end

# ============================================================
# 1. Family / engine data
# ============================================================

mutable struct PFEngine
    T
    K
    R
    xs
    weights::Vector{Int}
    f
    J
    params::Dict{Symbol, Any}
    dfparams::Dict{Symbol, Any}
    basis_numerators
    basis_pole_orders::Vector{Int}
    lift_fn::Function
end
function make_pf_engine(; weights=[1,2,1])
    T, (mu0, mu1, mu2, E) = polynomial_ring(QQ, [:mu0, :mu1, :mu2, :E])
    K = fraction_field(T)

    R, (x, y, z) = graded_polynomial_ring(K, [:x, :y, :z], weights)

    f = K(1//2)*y^2 +
        x^4 -
        mu2*x^3*z +
        (E - K(2))*x^2*z^2 +
        (mu1 + mu2)*x*z^3 +
        (mu0 - E + K(1))*z^4

    J = ideal(R, [derivative(f, x), derivative(f, y), derivative(f, z)])

    params = Dict(
        :mu0 => mu0,
        :mu1 => mu1,
        :mu2 => mu2,
        :E   => E,
    )

    # IMPORTANT: parameter derivatives in the coefficient field,
    # written directly as elements of R
    dfparams = Dict(
        :mu0 => z^4,
        :mu1 => x*z^3,
        :mu2 => -x^3*z + x*z^3,
        :E   => x^2*z^2 - z^4,
    )

    basis_numerators = [one(R), z^4]
    basis_pole_orders = [1, 2]

    eng = PFEngine(
        T, K, R, [x,y,z], weights, f, J,
        params, dfparams,
        basis_numerators, basis_pole_orders,
        (h, J) -> error("lift_fn not installed yet")
    )

    return eng
end

# ============================================================
# 2. Homogeneous lift h = sum q_i * J_i by linear solve
# ============================================================

function lift_jacobian_homogeneous(h, J; weights::Vector{Int} = [1,2,1])
    R  = base_ring(J)
    K  = base_ring(R)
    xs = gens(R)
    Js = gens(J)

    length(xs) == length(weights) || error("weights must match number of variables")

    D = weighted_degree_homogeneous(h, weights)
    D >= 0 || error("h must be nonzero and homogeneous")

    target_basis = weighted_monomials(xs, weights, D)
    isempty(target_basis) && error("No target monomials found in weighted degree $D")

    block_monomials = Vector{Vector{elem_type(R)}}()
    column_polys    = elem_type(R)[]

    for g in Js
        dg = weighted_degree_homogeneous(g, weights)
        mons = weighted_monomials(xs, weights, D - dg)
        push!(block_monomials, mons)
        append!(column_polys, [m * g for m in mons])
    end

    nrows = length(target_basis)
    ncols = length(column_polys)

    Adata = [coeff(column_polys[j], target_basis[i]) for i in 1:nrows, j in 1:ncols]
    bvec  = [coeff(h, target_basis[i]) for i in 1:nrows]

    A = matrix(K, Adata)
    ok, sol = can_solve_with_solution(A, bvec; side = :right)
    ok || error("Could not solve homogeneous lift system; h may not lie in J.")

    qs = elem_type(R)[]
    idx = 1
    for mons in block_monomials
        qi = zero(R)
        for m in mons
            qi += sol[idx] * m
            idx += 1
        end
        push!(qs, qi)
    end

    check = zero(R)
    for i in 1:length(Js)
        check += qs[i] * Js[i]
    end
    check == h || error("Internal error: reconstructed lift does not match h.")

    return qs
end

# ============================================================
# 3. Normal form + quotients
# ============================================================

struct NFData{T}
    remainder::T
    quotients::Vector{T}
end

function normal_form_with_quotients(p, J; lift_fn=nothing)
    R = base_ring(J)
    r = normal_form(p, J)
    h = p - r

    if iszero(h)
        return NFData(r, [zero(R) for _ in 1:length(gens(J))])
    end

    lift_fn === nothing && error("Need lift_fn(h, J) returning q with h = Σ q_i J_i.")
    q = lift_fn(h, J)

    @assert length(q) == length(gens(J))
    @assert h == sum(q[i] * gens(J)[i] for i in 1:length(gens(J)))

    return NFData(r, q)
end

# ============================================================
# 4. Griffiths–Dwork pole reduction
# ============================================================

function pole_reduce(p, f, J; lift_fn=nothing, weights=[1,2,1])
    nf = normal_form_with_quotients(p, J; lift_fn=lift_fn)

    xs   = gens(base_ring(J))
    degf = weighted_degree_homogeneous(f, weights)
    wdim = sum(weights)

    qdiv = zero(base_ring(J))
    for i in 1:length(xs)
        qdiv += derivative(nf.quotients[i], xs[i])
    end

    out = nf.remainder
    for comp in values(homogeneous_components(qdiv))
        if !iszero(comp)
            d = weighted_degree_homogeneous(comp, weights)
            out += comp / ((d + wdim) // degf)
        end
    end

    return out
end

function pole_reduce_full(p, f, J; lift_fn=nothing, weights=[1,2,1])
    degf = weighted_degree_homogeneous(f, weights)
    wdim = sum(weights)

    hc = homogeneous_components(p)
    isempty(hc) && return p

    max_pole_order = maximum(
        Int((weighted_degree_homogeneous(c, weights) + wdim) ÷ degf)
        for c in values(hc) if !iszero(c)
    )

    out = p
    for _ in 1:max_pole_order
        out = pole_reduce(out, f, J; lift_fn=lift_fn, weights=weights)
    end
    return out
end

# Turn whatever OSCAR gives back into an ordinary Julia vector
function unpack_solution(sol)
    # case 1: already iterable like a vector
    try
        v = collect(sol)
        isempty(v) || return v
    catch
    end

    # case 2: column matrix-like
    try
        return [sol[i, 1] for i in 1:nrows(sol)]
    catch
    end

    # case 3: row matrix-like
    try
        return [sol[1, j] for j in 1:ncols(sol)]
    catch
    end

    error("Don't know how to unpack solution of type $(typeof(sol))")
end


# ============================================================
# 5. Quotient-degree coordinates
# ============================================================

function quotient_degree_coords(eng::PFEngine, p, d::Int, preferred_basis)
    R = eng.R
    K = eng.K
    xs = eng.xs
    weights = eng.weights

    mons = weighted_monomials(xs, weights, d)
    isempty(mons) && error("No monomials in weighted degree $d")

    rp = normal_form(p, eng.J)
    rhs = [coeff(rp, m) for m in mons]

    cols = Vector{Vector{elem_type(K)}}()
    for b in preferred_basis
        rb = normal_form(b, eng.J)
        push!(cols, [coeff(rb, m) for m in mons])
    end

    A = matrix(K, hcat(cols...))
    ok, sol = can_solve_with_solution(A, rhs; side=:right)
    ok || error("Could not express degree-$d piece in preferred basis.")

    return unpack_solution(sol)
end


# ============================================================
# 6. Reduce a class to the basis {1, z^4}
# ============================================================


function reduce_to_basis(eng::PFEngine, p)
    r = normal_form(p, eng.J)

    p0 = homogeneous_piece_of_degree(r, eng.weights, 0)
    p4 = homogeneous_piece_of_degree(r, eng.weights, 4)

    extra = zero(eng.R)
    for comp in values(homogeneous_components(r))
        if !iszero(comp)
            d = weighted_degree_homogeneous(comp, eng.weights)
            if d != 0 && d != 4
                extra += comp
            end
        end
    end
    iszero(extra) || error("Reduction still contains unsupported degrees; got extra piece $extra")

    c0 = iszero(p0) ? zero(eng.K) : coeff(p0, one(eng.R))

    c4 = if iszero(p4)
        zero(eng.K)
    else
        v = quotient_degree_coords(eng, p4, 4, [eng.basis_numerators[2]])
        length(v) == 1 || error("Expected a 1-dimensional degree-4 coordinate vector, got $v")
        v[1]
    end

    return (c0=c0, c4=c4)
end


function debug_reduce_to_basis(eng::PFEngine, p)
    r = normal_form(p, eng.J)
    println("input        = ", p)
    println("normal form  = ", r)
    println("deg-0 piece  = ", homogeneous_piece_of_degree(r, eng.weights, 0))
    println("deg-4 piece  = ", homogeneous_piece_of_degree(r, eng.weights, 4))
    println("basis coords = ", reduce_to_basis(eng, p))
end

# ============================================================
# 7. Differentiate basis classes and build the GM/PF matrix
# ============================================================

# For now basis numerators are parameter-independent, so only the pole term contributes:
# ∂_λ [P Ω / f^k] = [(f ∂_λ P - k P ∂_λ f) Ω / f^(k+1)]
function differentiate_basis_class(eng::PFEngine, idx::Int, param::Symbol)
    P  = eng.basis_numerators[idx]
    k  = eng.basis_pole_orders[idx]
    df = eng.dfparams[param]

    num = -k * P * df
    red = pole_reduce_full(num, eng.f, eng.J; lift_fn=eng.lift_fn, weights=eng.weights)
    return red
end

# Row-convention:
# row i = coordinates of ∂(basis_i) in the chosen basis.
# Then for period column vector Π = (∮basis_1, ∮basis_2)^T, we have Π' = M Π.

function connection_matrix(eng::PFEngine, param::Symbol)
    nb = length(eng.basis_numerators)
    data = [zero(eng.K) for _ in 1:nb, _ in 1:nb]

    for i in 1:nb
        red = differentiate_basis_class(eng, i, param)
        coords = reduce_to_basis(eng, red)
        data[i, 1] = coords.c0
        data[i, 2] = coords.c4
    end

    return matrix(eng.K, data)
end

# ============================================================
# 8. Scalar 2nd-order PF equation from a 2x2 system
# ============================================================

# tries derivative on coefficient-field elements
function diff_coeff(a, t)
    return derivative(a, t)
end

# If Π' = M Π with M = [a b; c d], then u = Π1 satisfies
# u'' + A1 u' + A0 u = 0
# where
# A1 = -(a + d + b'/b)
# A0 =  a*d - b*c - a' + a b'/b
function scalar_pf_coeffs(M, t)
    a, b = M[1,1], M[1,2]
    c, d = M[2,1], M[2,2]

    bp = diff_coeff(b, t)
    ap = diff_coeff(a, t)

    A1 = -(a + d + bp / b)
    A0 = a*d - b*c - ap + a*bp / b

    return A1, A0
end




# Pretty printer
function show_pf_equation(A1, A0; u="u", var="t")
    println("Picard–Fuchs equation:")
    println("$u''($var) + ($A1) * $u'($var) + ($A0) * $u($var) = 0")
end

# ============================================================
# 9. Build engine + install lift + smoke tests
# ============================================================


eng = make_pf_engine()
eng.lift_fn = (h, J) -> lift_jacobian_homogeneous(h, J; weights=eng.weights)

println("f = ", eng.f)
println("Jacobian generators:")
for g in gens(eng.J)
    println("  ", g)
end

println("\nSmoke test: normal form with quotients on x^8")
nf = normal_form_with_quotients(eng.xs[1]^8, eng.J; lift_fn=eng.lift_fn)
println("remainder = ", nf.remainder)
println("check     = ", eng.xs[1]^8 == nf.remainder + sum(nf.quotients[i]*gens(eng.J)[i] for i in 1:length(gens(eng.J))))

println("\nSmoke test: z^4")
rb = reduce_to_basis(eng, eng.xs[3]^4)
println("coords(z^4) = ", rb)

println("\nDetailed debug for z^4:")
debug_reduce_to_basis(eng, eng.xs[3]^4)


#println("\nConnection matrix in E-direction:")
#ME = connection_matrix(eng, :E)
#println(ME)

#println("\nScalar PF coefficients for the first period:")
#A1, A0 = scalar_pf_coeffs(ME, eng.params[:E])
#show_pf_equation(A1, A0; u="Π1", var="E")

f = x^4 - mu2*x^3*z + (E - 2)*x^2*z^2 + (mu1 + mu2)*x*z^3 + 1//2*y^2 + (mu0 - E + 1)*z^4
Jacobian generators:
  4*x^3 - 3*mu2*x^2*z + (2*E - 4)*x*z^2 + (mu1 + mu2)*z^3
  y
  -mu2*x^3 + (2*E - 4)*x^2*z + (3*mu1 + 3*mu2)*x*z^2 + (4*mu0 - 4*E + 4)*z^3

Smoke test: normal form with quotients on x^8
remainder = 0
check     = true

Smoke test: z^4
coords(z^4) = (c0 = 0, c4 = 1)

Detailed debug for z^4:
input        = z^4
normal form  = z^4
deg-0 piece  = 0
deg-4 piece  = z^4
basis coords = (c0 = 0, c4 = 1)


In [24]:
# sanity check for the first basis vector too
println("coords(1) = ", reduce_to_basis(eng, one(eng.R)))
debug_reduce_to_basis(eng, one(eng.R))

coords(1) = (c0 = 1, c4 = 0)
input        = 1
normal form  = 1
deg-0 piece  = 1
deg-4 piece  = 0
basis coords = (c0 = 1, c4 = 0)


In [25]:
function debug_derivative_class(eng::PFEngine, idx::Int, param::Symbol)
    P  = eng.basis_numerators[idx]
    k  = eng.basis_pole_orders[idx]
    df = eng.dfparams[param]

    num = -k * P * df
    red = pole_reduce_full(num, eng.f, eng.J; lift_fn=eng.lift_fn, weights=eng.weights)
    coords = reduce_to_basis(eng, red)

    println("basis index   = ", idx)
    println("parameter     = ", param)
    println("basis num P   = ", P)
    println("pole order k  = ", k)
    println("df_param      = ", df)
    println("raw numerator = ", num)
    println("reduced       = ", red)
    println("coords        = ", coords)
    println()
end

function debug_derivative_class_compact(eng::PFEngine, idx::Int, param::Symbol; show_reduced=false)
    P  = eng.basis_numerators[idx]
    k  = eng.basis_pole_orders[idx]
    df = eng.dfparams[param]

    num = -k * P * df
    red = pole_reduce_full(num, eng.f, eng.J; lift_fn=eng.lift_fn, weights=eng.weights)
    coords = reduce_to_basis(eng, red)

    println("basis index   = ", idx)
    println("parameter     = ", param)
    println("raw numerator = ", num)
    if show_reduced
        println("reduced       = ", red)
    else
        println("reduced degs  = ",
            [weighted_degree_homogeneous(c, eng.weights)
             for c in values(homogeneous_components(red)) if !iszero(c)])
    end
    println("coords        = ", coords)
    println()
end


debug_derivative_class_compact (generic function with 1 method)

In [26]:
for p in (:mu0, :mu1, :mu2, :E)
    println("========================================")
    debug_derivative_class_compact(eng, 1, p)
   # debug_derivative_class(eng, 2, p)
end

basis index   = 1
parameter     = mu0
raw numerator = -z^4
reduced degs  = [4]
coords        = (c0 = 0, c4 = -1)

basis index   = 1
parameter     = mu1
raw numerator = -x*z^3
reduced degs  = [4, 0]
coords        = (c0 = (-mu1 + 1//8*mu2^3 - 1//2*mu2*E)//(mu0*mu2^2 - 8//3*mu0*E + 16//3*mu0 + 3*mu1^2 - 1//2*mu1*mu2^3 + 7//3*mu1*mu2*E + 4//3*mu1*mu2 - 1//2*mu2^4 - 1//6*mu2^2*E^2 + 2*mu2^2*E - 4//3*mu2^2 + 2//3*E^3 - 4//3*E^2), c4 = (4*mu0*mu1 - 3//4*mu0*mu2^3 + 8//3*mu0*mu2*E - 4//3*mu0*mu2 + 1//4*mu1^2*mu2 - 1//12*mu1*mu2^2*E + 2//3*mu1*mu2^2 + 1//3*mu1*E^2 - 16//3*mu1*E + 16//3*mu1 + 2//3*mu2^3*E - 1//3*mu2^3 - 7//3*mu2*E^2 + 8//3*mu2*E)//(mu0*mu2^2 - 8//3*mu0*E + 16//3*mu0 + 3*mu1^2 - 1//2*mu1*mu2^3 + 7//3*mu1*mu2*E + 4//3*mu1*mu2 - 1//2*mu2^4 - 1//6*mu2^2*E^2 + 2*mu2^2*E - 4//3*mu2^2 + 2//3*E^3 - 4//3*E^2))

basis index   = 1
parameter     = mu2
raw numerator = x^3*z - x*z^3
reduced degs  = [4, 0]
coords        = (c0 = (-mu0*mu2 + 1//8*mu1*mu2^2 - 1//2*mu1*E + 1//4*mu2^3)//(mu0*mu2^2 

In [ ]:
for p in (:mu0, :mu1, :mu2, :E)
    println("========================================")
    println("Connection matrix for ", p)
    println(connection_matrix(eng, p))
    println()
end

Connection matrix for mu0


## optimizations

In [27]:
# Convert an OSCAR row/column matrix to a Julia Vector
function matrix_to_vector(M)
    r = nrows(M)
    c = ncols(M)

    if c == 1
        return [M[i, 1] for i in 1:r]
    elseif r == 1
        return [M[1, j] for j in 1:c]
    else
        error("Expected a row or column matrix, got size ($r,$c).")
    end
end

matrix_to_vector (generic function with 1 method)

In [41]:
# Precompute GB data for one ideal J
function gb_lift_data(J)
    gb, C = groebner_basis_with_transformation_matrix(J)
    return (gb=gb, C=C, Jgens=gens(J))
end

function lift_via_gb_data(p, data)
    q, r = reduce_with_quotients(p, data.gb)

    #  try the lift implementation suggested by Max
    Aq = data.C * transpose(q)

    a_vec = matrix_to_vector(Aq)

    return (quotients = a_vec, remainder = r, gb_quotients = q, lifted_matrix = Aq)
end

# Reconstruct sum_i a_i J_i
function reconstruct_from_lift(a_vec, Jgens)
    s = zero(parent(Jgens[1]))
    for i in 1:length(Jgens)
        s += a_vec[i] * Jgens[i]
    end
    return s
end

reconstruct_from_lift (generic function with 1 method)

In [42]:
# test the lift

data = gb_lift_data(eng.J)

testp = eng.xs[1]^8
lift = lift_via_gb_data(testp, data)

lhs = reconstruct_from_lift(lift.quotients, data.Jgens) + lift.remainder

@assert lhs == testp

In [34]:
function make_gb_lift_fn(J)
    data = gb_lift_data(J)

    function lift_fn(p, Jdummy)
        q, r = reduce_with_quotients(p, data.gb)
        iszero(r) || error("Nonzero remainder in GB lift.")

        a_mat = data.C * transpose(q)
        a_vec = matrix_to_vector(a_mat)

        return a_vec
    end

    return lift_fn
end

make_gb_lift_fn (generic function with 1 method)

In [37]:
# 1) GB data + lift
dataJ = gb_lift_data(eng.J)
eng.lift_fn = make_gb_lift_fn(eng.J)

(::var"#lift_fn#make_gb_lift_fn##0"{@NamedTuple{gb::Oscar.IdealGens{MPolyDecRingElem{AbstractAlgebra.Generic.FracFieldElem{QQMPolyRingElem}, AbstractAlgebra.Generic.MPoly{AbstractAlgebra.Generic.FracFieldElem{QQMPolyRingElem}}}}, C::AbstractAlgebra.Generic.MatSpaceElem{MPolyDecRingElem{AbstractAlgebra.Generic.FracFieldElem{QQMPolyRingElem}, AbstractAlgebra.Generic.MPoly{AbstractAlgebra.Generic.FracFieldElem{QQMPolyRingElem}}}}, Jgens::Vector{MPolyDecRingElem{AbstractAlgebra.Generic.FracFieldElem{QQMPolyRingElem}, AbstractAlgebra.Generic.MPoly{AbstractAlgebra.Generic.FracFieldElem{QQMPolyRingElem}}}}}}) (generic function with 1 method)

In [36]:
x, y, z = eng.xs

println(normal_form_with_quotients(x^8, eng.J; lift_fn=eng.lift_fn))
println(pole_reduce(x^8, eng.f, eng.J; lift_fn=eng.lift_fn, weights=eng.weights))

NFData{MPolyDecRingElem{AbstractAlgebra.Generic.FracFieldElem{QQMPolyRingElem}, AbstractAlgebra.Generic.MPoly{AbstractAlgebra.Generic.FracFieldElem{QQMPolyRingElem}}}}(0, MPolyDecRingElem{AbstractAlgebra.Generic.FracFieldElem{QQMPolyRingElem}, AbstractAlgebra.Generic.MPoly{AbstractAlgebra.Generic.FracFieldElem{QQMPolyRingElem}}}[(-2//3*E + 4//3)//(mu2^2 - 8//3*E + 16//3)*x^5 + (mu0^4*mu2^2 + 5//4*mu0^3*mu1*mu2^3 + 5//3*mu0^3*mu1*mu2*E - 10//3*mu0^3*mu1*mu2 + 3//8*mu0^3*mu2^4*E + 1//2*mu0^3*mu2^4 - 1//4*mu0^3*mu2^2*E^2 - 4//3*mu0^3*mu2^2*E - 1//3*mu0^3*mu2^2 + 2//3*mu0^3*E^3 - 4*mu0^3*E^2 + 8*mu0^3*E - 16//3*mu0^3 + 91//256*mu0^2*mu1^2*mu2^4 + 173//96*mu0^2*mu1^2*mu2^2*E - 173//48*mu0^2*mu1^2*mu2^2 + 91//128*mu0^2*mu1*mu2^5 + 35//32*mu0^2*mu1*mu2^3*E^2 - 217//48*mu0^2*mu1*mu2^3*E + 11//12*mu0^2*mu1*mu2^3 - 1//3*mu0^2*mu1*mu2*E^3 - 3*mu0^2*mu1*mu2*E^2 + 11*mu0^2*mu1*mu2*E - 22//3*mu0^2*mu1*mu2 + 91//256*mu0^2*mu2^6 - 1//32*mu0^2*mu2^4*E^2 - 283//96*mu0^2*mu2^4*E + 109//48*mu0^2*mu2^4 + 1

In [44]:
eng.dfparams = Dict(
    :E   => x^2*z^2 - z^4,
    :mu0 => z^4,
    :mu1 => x*z^3,
    :mu2 => -x^3*z + x*z^3
)

Dict{Symbol, MPolyDecRingElem{AbstractAlgebra.Generic.FracFieldElem{QQMPolyRingElem}, AbstractAlgebra.Generic.MPoly{AbstractAlgebra.Generic.FracFieldElem{QQMPolyRingElem}}}} with 4 entries:
  :mu0 => z^4
  :mu1 => x*z^3
  :mu2 => -x^3*z + x*z^3
  :E   => x^2*z^2 - z^4

In [39]:
# Reduce a numerator P to the basis {Ω/f, z^4 Ω/f^2}
function reduce_deg4_class(eng::PFEngine, P)
    red = pole_reduce_full(P, eng.f, eng.J;
                           lift_fn=eng.lift_fn,
                           weights=eng.weights)
    return reduce_to_basis(eng, red)
end

# j=1 corresponds to Ω/f, j=2 to z^4 Ω/f^2
function gm_column(eng::PFEngine, λ::Symbol, j::Int)
    P = eng.basis_numerators[j]
    k = eng.basis_pole_orders[j]

    num = -k * P * eng.dfparams[λ]
    red = pole_reduce_full(num, eng.f, eng.J;
                           lift_fn=eng.lift_fn,
                           weights=eng.weights)
    coords = reduce_to_basis(eng, red)

    return (num=num, red=red, c0=coords.c0, c4=coords.c4)
end

function gm_matrix(eng::PFEngine, λ::Symbol)
    c1 = gm_column(eng, λ, 1)
    c2 = gm_column(eng, λ, 2)
    A  = [c1.c0 c2.c0;
          c1.c4 c2.c4]
    return (col1=c1, col2=c2, A=A)
end

gm_matrix (generic function with 1 method)

In [40]:
AE = gm_matrix(eng, :E)
println("A_E = ")
println(AE.A)

println("col1 numerator = ", AE.col1.num)
println("col2 numerator = ", AE.col2.num)

A_E = 
AbstractAlgebra.Generic.FracFieldElem{QQMPolyRingElem}[(4//3*mu0 - 1//6*mu1*mu2 + 1//12*mu2^2*E - 1//3*mu2^2 - 1//3*E^2)//(mu0*mu2^2 - 8//3*mu0*E + 16//3*mu0 + 3*mu1^2 - 1//2*mu1*mu2^3 + 7//3*mu1*mu2*E + 4//3*mu1*mu2 - 1//2*mu2^4 - 1//6*mu2^2*E^2 + 2*mu2^2*E - 4//3*mu2^2 + 2//3*E^3 - 4//3*E^2) (mu0^3 - 1//8*mu0^2*mu1*mu2 + 1//64*mu0^2*mu2^4 + 3//32*mu0^2*mu2^2*E - 1//2*mu0^2*mu2^2 - 1//2*mu0^2*E^2 - 1//2*mu0^2*E - 1//64*mu0*mu1^2*mu2^2 - 13//32*mu0*mu1^2*E - 1//2*mu0*mu1^2 - 1//256*mu0*mu1*mu2^5 + 17//128*mu0*mu1*mu2^3*E - 1//16*mu0*mu1*mu2^3 - 15//32*mu0*mu1*mu2*E^2 + 1//4*mu0*mu1*mu2*E - mu0*mu1*mu2 - 15//2048*mu0*mu2^6*E + 1//128*mu0*mu2^6 + 1//16*mu0*mu2^4*E^2 - 1//8*mu0*mu2^4*E + 3//16*mu0*mu2^4 - 19//128*mu0*mu2^2*E^3 + 1//4*mu0*mu2^2*E^2 - 1//4*mu0*mu2^2*E + 1//16*mu0*E^4 + 1//4*mu0*E^3 + 9//64*mu1^4 - 17//512*mu1^3*mu2^3 + 17//128*mu1^3*mu2*E - 1//32*mu1^3*mu2 + 5//2048*mu1^2*mu2^6 - 43//2048*mu1^2*mu2^4*E + 5//128*mu1^2*mu2^4 + 13//256*mu1^2*mu2^2*E^2 - 29//128*mu1^2*mu

In [53]:
Aμ2 = gm_matrix(eng, :mu2)
println("A_mu2 = ")
println(Aμ2.A)

A_mu2 = 
AbstractAlgebra.Generic.FracFieldElem{QQMPolyRingElem}[(-mu0*mu2 + 1//8*mu1*mu2^2 - 1//2*mu1*E + 1//4*mu2^3)//(mu0*mu2^2 - 8//3*mu0*E + 16//3*mu0 + 3*mu1^2 - 1//2*mu1*mu2^3 + 7//3*mu1*mu2*E + 4//3*mu1*mu2 - 1//2*mu2^4 - 1//6*mu2^2*E^2 + 2*mu2^2*E - 4//3*mu2^2 + 2//3*E^3 - 4//3*E^2) (-3//4*mu0^3*mu2 + 9//64*mu0^2*mu1*mu2^2 - mu0^2*mu1*E + 1//4*mu0^2*mu1 - 3//256*mu0^2*mu2^5 + 1//16*mu0^2*mu2^3*E + 9//32*mu0^2*mu2^3 - 1//8*mu0^2*mu2*E^2 + 3//4*mu0^2*mu2*E - 1//2*mu0^2*mu2 + 21//64*mu0*mu1^3 - 3//64*mu0*mu1^2*mu2^3 + 19//64*mu0*mu1^2*mu2*E - 5//32*mu0*mu1^2*mu2 + 15//4096*mu0*mu1*mu2^6 - 17//512*mu0*mu1*mu2^4*E + 15//128*mu0*mu1*mu2^4 + 3//256*mu0*mu1*mu2^2*E^2 - 13//32*mu0*mu1*mu2^2*E + 3//8*mu0*mu1*mu2^2 + 1//4*mu0*mu1*E^3 + 7//8*mu0*mu1*E^2 - mu0*mu1*E - 15//2048*mu0*mu2^7 + 5//1024*mu0*mu2^5*E^2 + 19//256*mu0*mu2^5*E - 3//64*mu0*mu2^5 - 5//128*mu0*mu2^3*E^3 - 23//128*mu0*mu2^3*E^2 - 1//8*mu0*mu2^3*E + 1//4*mu0*mu2^3 + 5//64*mu0*mu2*E^4 + 1//8*mu0*mu2*E^3 - 1//4*mu0*mu2*E^2 - 

In [50]:
Aμ1 = gm_matrix(eng, :mu1)
println("A_mu1 = ")
println(Aμ1.A)

A_mu1 = 
AbstractAlgebra.Generic.FracFieldElem{QQMPolyRingElem}[(-mu1 + 1//8*mu2^3 - 1//2*mu2*E)//(mu0*mu2^2 - 8//3*mu0*E + 16//3*mu0 + 3*mu1^2 - 1//2*mu1*mu2^3 + 7//3*mu1*mu2*E + 4//3*mu1*mu2 - 1//2*mu2^4 - 1//6*mu2^2*E^2 + 2*mu2^2*E - 4//3*mu2^2 + 2//3*E^3 - 4//3*E^2) (-7//4*mu0^2*mu1 + 17//64*mu0^2*mu2^3 - mu0^2*mu2*E + 1//4*mu0^2*mu2 - 35//64*mu0*mu1^2*mu2 + 37//256*mu0*mu1*mu2^4 - 43//64*mu0*mu1*mu2^2*E + 1//4*mu0*mu1*mu2^2 + 3//8*mu0*mu1*E^2 + 2*mu0*mu1*E - 2*mu0*mu1 - 45//4096*mu0*mu2^7 + 53//512*mu0*mu2^5*E - 1//16*mu0*mu2^5 - 77//256*mu0*mu2^3*E^2 + 1//8*mu0*mu2^3 + 1//4*mu0*mu2*E^3 + 7//8*mu0*mu2*E^2 - mu0*mu2*E + 1//128*mu1^3*mu2^2 - 3//32*mu1^3*E + 3//16*mu1^3 - 1//4096*mu1^2*mu2^5 + 9//512*mu1^2*mu2^3*E - 3//256*mu1^2*mu2^3 - 17//256*mu1^2*mu2*E^2 + 17//32*mu1^2*mu2*E - 1//4*mu1^2*mu2 - 5//4096*mu1*mu2^6*E + 1//512*mu1*mu2^6 + 11//1024*mu1*mu2^4*E^2 - 39//256*mu1*mu2^4*E + 9//64*mu1*mu2^4 - 7//256*mu1*mu2^2*E^3 + 45//64*mu1*mu2^2*E^2 - mu1*mu2^2*E + 1//2*mu1*mu2^2 + 1//64*

In [51]:
Aμ0 = gm_matrix(eng, :mu0)
println("A_mu0 = ")
println(Aμ0.A)

A_mu0 = 
AbstractAlgebra.Generic.FracFieldElem{QQMPolyRingElem}[0 (3//16*mu0^2*mu2^2 - 1//2*mu0^2*E + mu0^2 + 21//16*mu0*mu1^2 - 15//64*mu0*mu1*mu2^3 + 17//16*mu0*mu1*mu2*E + 1//2*mu0*mu1*mu2 + 3//1024*mu0*mu2^6 - 3//128*mu0*mu2^4*E - 3//16*mu0*mu2^4 - 1//64*mu0*mu2^2*E^2 + 3//4*mu0*mu2^2*E - 1//2*mu0*mu2^2 + 1//4*mu0*E^3 - 1//2*mu0*E^2 + 21//64*mu1^3*mu2 - 99//1024*mu1^2*mu2^4 + 7//16*mu1^2*mu2^2*E + 7//64*mu1^2*mu2^2 - 5//64*mu1^2*E^2 - mu1^2*E + mu1^2 + 15//2048*mu1*mu2^7 - 69//1024*mu1*mu2^5*E - 15//256*mu1*mu2^5 + 5//32*mu1*mu2^3*E^2 + 31//64*mu1*mu2^3*E - 3//8*mu1*mu2^3 - 1//64*mu1*mu2*E^3 - 9//8*mu1*mu2*E^2 + mu1*mu2*E + 15//2048*mu2^8 + 5//2048*mu2^6*E^2 - 41//512*mu2^6*E + 13//256*mu2^6 - 11//512*mu2^4*E^3 + 79//256*mu2^4*E^2 - 9//32*mu2^4*E + 1//16*mu2^4 + 7//128*mu2^2*E^4 - 7//16*mu2^2*E^3 + 3//8*mu2^2*E^2 - 1//32*E^5 + 1//16*E^4)//(mu0^4*mu2^2 - 8//3*mu0^4*E + 16//3*mu0^4 + 3*mu0^3*mu1^2 + 1//4*mu0^3*mu1*mu2^3 + 1//3*mu0^3*mu1*mu2*E + 16//3*mu0^3*mu1*mu2 - 27//256*mu0^3*mu2

In [54]:
vars = [eng.params[:mu0], eng.params[:mu1], eng.params[:mu2], eng.params[:E]]
vals = [1//7, 1//9, 1//5, 3//4]   # example

function specialize_coeff(c, vars, vals)
    num = numerator(c)
    den = denominator(c)

    numv = evaluate(num, vars, vals)
    denv = evaluate(den, vars, vals)

    iszero(denv) && error("Specialization hit a pole: denominator vanished.")

    return numv // denv
end
function specialize_matrix(M, vars, vals)
    [specialize_coeff(M[i,j], vars, vals) for i in 1:size(M,1), j in 1:size(M,2)]
end

AEv  = specialize_matrix(AE.A,  vars, vals)
A0v  = specialize_matrix(Aμ0.A, vars, vals)
A1v  = specialize_matrix(Aμ1.A, vars, vals)
A2v  = specialize_matrix(Aμ2.A, vars, vals)

println("A_E  @ point = ", AEv)
println("A_mu0 @ point = ", A0v)
println("A_mu1 @ point = ", A1v)
println("A_mu2 @ point = ", A2v)

A_E  @ point = AbstractAlgebra.Generic.FracFieldElem{QQMPolyRingElem}[-21850//227523 100607038135506000//21628998870019409; 40571887//28667898 -157457544766975550//21628998870019409]
A_mu0 @ point = AbstractAlgebra.Generic.FracFieldElem{QQMPolyRingElem}[0 -281486523207204000//21628998870019409; -1 557219626951362900//21628998870019409]
A_mu1 @ point = AbstractAlgebra.Generic.FracFieldElem{QQMPolyRingElem}[-116620//75841 -13497467556636000//21628998870019409; 2227175//682569 23448705600709340//21628998870019409]
A_mu2 @ point = AbstractAlgebra.Generic.FracFieldElem{QQMPolyRingElem}[-42640//75841 176976177567000//21628998870019409; 20708455//19111932 -23495230226858290//64886996610058227]


In [55]:
mu0T, mu1T, mu2T, ET = gens(eng.T)
varsT = [mu0T, mu1T, mu2T, ET]
vals  = [1//7, 2//9, 1//5, 3//4]   # example point

4-element Vector{Rational{Int64}}:
 1//7
 2//9
 1//5
 3//4

In [56]:
function specialize_coeff_T(c, varsT, vals)
    num = numerator(c)
    den = denominator(c)

    numv = evaluate(num, varsT, vals)
    denv = evaluate(den, varsT, vals)

    iszero(denv) && error("Specialization hit a pole.")
    return numv // denv
end

function specialize_matrix_T(M, varsT, vals)
    [specialize_coeff_T(M[i,j], varsT, vals) for i in 1:size(M,1), j in 1:size(M,2)]
end

specialize_matrix_T (generic function with 1 method)

In [57]:
AEv  = specialize_matrix_T(AE.A,  varsT, vals)
A0v  = specialize_matrix_T(Aμ0.A, varsT, vals)
A1v  = specialize_matrix_T(Aμ1.A, varsT, vals)
A2v  = specialize_matrix_T(Aμ2.A, varsT, vals)

println("A_E  @ point = ", AEv)
println("A_mu0 @ point = ", A0v)
println("A_mu1 @ point = ", A1v)
println("A_mu2 @ point = ", A2v)

A_E  @ point = AbstractAlgebra.Generic.FracFieldElem{QQMPolyRingElem}[-28850//566183 753732439194798000//428387498581863887; 94113907//71339058 -509372324153653650//428387498581863887]
A_mu0 @ point = AbstractAlgebra.Generic.FracFieldElem{QQMPolyRingElem}[0 -2202943607726652000//428387498581863887; -1 3854909868596696700//428387498581863887]
A_mu1 @ point = AbstractAlgebra.Generic.FracFieldElem{QQMPolyRingElem}[-559860//566183 -138594869504028000//428387498581863887; 3571025//1698549 160445742637232220//428387498581863887]
A_mu2 @ point = AbstractAlgebra.Generic.FracFieldElem{QQMPolyRingElem}[-205620//566183 -67124337155379000//428387498581863887; 90594365//142678116 -34756325381176390//428387498581863887]


In [58]:
println("A_mu0 first column = ", (Aμ0.A[1,1], Aμ0.A[2,1]))

A_mu0 first column = (0, -1)


In [59]:
function dcoeff(c, p)
    num = numerator(c)
    den = denominator(c)
    return (derivative(num, p) * den - num * derivative(den, p)) // den^2
end

function dmat(A, p)
    [dcoeff(A[i,j], p) for i in 1:size(A,1), j in 1:size(A,2)]
end

comm(A, B) = A * B - B * A

function curvature(A1, A2, p1, p2)
    dmat(A2, p1) .- dmat(A1, p2) .+ comm(A1, A2)
end

curvature (generic function with 1 method)

In [60]:
F_E_mu0  = curvature(AE.A,  Aμ0.A, ET,   mu0T)
F_E_mu1  = curvature(AE.A,  Aμ1.A, ET,   mu1T)
F_E_mu2  = curvature(AE.A,  Aμ2.A, ET,   mu2T)
F_mu0mu1 = curvature(Aμ0.A, Aμ1.A, mu0T, mu1T)
F_mu0mu2 = curvature(Aμ0.A, Aμ2.A, mu0T, mu2T)
F_mu1mu2 = curvature(Aμ1.A, Aμ2.A, mu1T, mu2T)

2×2 Matrix{AbstractAlgebra.Generic.FracFieldElem{QQMPolyRingElem}}:
 0  0
 0  0

In [61]:
F_E_mu0_v  = specialize_matrix_T(F_E_mu0,  varsT, vals)
F_E_mu1_v  = specialize_matrix_T(F_E_mu1,  varsT, vals)
F_E_mu2_v  = specialize_matrix_T(F_E_mu2,  varsT, vals)
F_mu0mu1_v = specialize_matrix_T(F_mu0mu1, varsT, vals)
F_mu0mu2_v = specialize_matrix_T(F_mu0mu2, varsT, vals)
F_mu1mu2_v = specialize_matrix_T(F_mu1mu2, varsT, vals)

println("F(E,mu0)  @ point = ", F_E_mu0_v)
println("F(E,mu1)  @ point = ", F_E_mu1_v)
println("F(E,mu2)  @ point = ", F_E_mu2_v)
println("F(mu0,mu1)@ point = ", F_mu0mu1_v)
println("F(mu0,mu2)@ point = ", F_mu0mu2_v)
println("F(mu1,mu2)@ point = ", F_mu1mu2_v)

F(E,mu0)  @ point = AbstractAlgebra.Generic.FracFieldElem{QQMPolyRingElem}[0 0; 0 0]
F(E,mu1)  @ point = AbstractAlgebra.Generic.FracFieldElem{QQMPolyRingElem}[0 0; 0 0]
F(E,mu2)  @ point = AbstractAlgebra.Generic.FracFieldElem{QQMPolyRingElem}[0 0; 0 0]
F(mu0,mu1)@ point = AbstractAlgebra.Generic.FracFieldElem{QQMPolyRingElem}[0 0; 0 0]
F(mu0,mu2)@ point = AbstractAlgebra.Generic.FracFieldElem{QQMPolyRingElem}[0 0; 0 0]
F(mu1,mu2)@ point = AbstractAlgebra.Generic.FracFieldElem{QQMPolyRingElem}[0 0; 0 0]


In [62]:
# Derivative of a fraction-field entry with respect to a parameter p in eng.T
function dcoeff(c, p)
    num = numerator(c)
    den = denominator(c)
    return (derivative(num, p) * den - num * derivative(den, p)) // den^2
end

# PF equation for the first component of Y' = A Y
# Returns P,Q such that y1'' = P*y1' + Q*y1
function pf_from_gm_first(AE, Epar)
    α = AE[1,1]
    β = AE[1,2]
    γ = AE[2,1]
    δ = AE[2,2]

    iszero(β) && error("Cannot eliminate second component: AE[1,2] = 0.")

    βE = dcoeff(β, Epar)

    P = α + δ + βE // β
    Q = dcoeff(α, Epar) + β*γ - α*δ - α*(βE // β)

    return (P=P, Q=Q)
end

# PF equation for the second component of Y' = A Y
# Returns P,Q such that y2'' = P*y2' + Q*y2
function pf_from_gm_second(AE, Epar)
    α = AE[1,1]
    β = AE[1,2]
    γ = AE[2,1]
    δ = AE[2,2]

    iszero(γ) && error("Cannot eliminate first component: AE[2,1] = 0.")

    γE = dcoeff(γ, Epar)

    P = α + δ + γE // γ
    Q = dcoeff(δ, Epar) + β*γ - α*δ - δ*(γE // γ)

    return (P=P, Q=Q)
end

pf_from_gm_second (generic function with 1 method)

In [63]:
mu0T, mu1T, mu2T, ET = gens(eng.T)

PF1 = pf_from_gm_first(AE.A, ET)
PF2 = pf_from_gm_second(AE.A, ET)

println("First component PF:")
println("P1 = ", PF1.P)
println("Q1 = ", PF1.Q)

#
# # first component
# y1'' = PF1.P * y1' + PF1.Q * y1

# second component
# y2'' = PF2.P * y2' + PF2.Q * y2
#

println("Second component PF:")
println("P2 = ", PF2.P)
println("Q2 = ", PF2.Q)

First component PF:
P1 = (-33//32*mu0^5*mu2^2 + mu0^5*E + 3//2*mu0^5 - 49//32*mu0^4*mu1^2 + 31//64*mu0^4*mu1*mu2^3 - 51//16*mu0^4*mu1*mu2*E + 23//8*mu0^4*mu1*mu2 - 285//8192*mu0^4*mu2^6 + 107//512*mu0^4*mu2^4*E + 105//512*mu0^4*mu2^4 - 27//128*mu0^4*mu2^2*E^2 + 25//8*mu0^4*mu2^2*E - 95//32*mu0^4*mu2^2 - 3//4*mu0^4*E^3 - 3*mu0^4*E^2 + 1//2*mu0^4*E - 1//2*mu0^4 + 7//64*mu0^3*mu1^3*mu2 + 833//8192*mu0^3*mu1^2*mu2^4 - 75//256*mu0^3*mu1^2*mu2^2*E + 285//256*mu0^3*mu1^2*mu2^2 - 75//128*mu0^3*mu1^2*E^2 + 25//8*mu0^3*mu1^2*E - 139//32*mu0^3*mu1^2 - 423//32768*mu0^3*mu1*mu2^7 + 199//2048*mu0^3*mu1*mu2^5*E - 55//1024*mu0^3*mu1*mu2^5 - 15//64*mu0^3*mu1*mu2^3*E^2 + 25//128*mu0^3*mu1*mu2^3*E + 29//128*mu0^3*mu1*mu2^3 + 3//32*mu0^3*mu1*mu2*E^3 + 165//32*mu0^3*mu1*mu2*E^2 - 41//4*mu0^3*mu1*mu2*E + 405//524288*mu0^3*mu2^10 - 297//32768*mu0^3*mu2^8*E + 45//16384*mu0^3*mu2^8 + 1227//32768*mu0^3*mu2^6*E^2 + 21//1024*mu0^3*mu2^6*E + 25//2048*mu0^3*mu2^6 - 49//1024*mu0^3*mu2^4*E^3 - 117//1024*mu0^3*mu2^4*E

### Begin old stuff

In [ ]:

# By default we assume y is the 2nd generator in the ring ordering (x,y,z).
# This is used to decide whether we can apply the faster "y-free" lift.

function is_yfree(p, y_index::Int=2)
    for i in 1:length(p)
        if exponent_vector(p, i)[y_index] != 0
            return false
        end
    end
    # Return true if the polynomial p contains no powers of the y-variable.
    return true
end


# Build a cached homogeneous lift specialized to the y-free sector.
#
# This is valid in the weighted P(1,2,1) model because ∂_y f = y, so y lies in
# the Jacobian ideal.  Therefore, if h is y-free, we can solve the lift problem
# using only the x- and z-generators fx and fz.
#
# This version uses the full q1/q3 ansatz and therefore gives an underdetermined
# system in some degrees; it is useful conceptually, but the square-gauge version
# below is usually faster and more stable.

function make_yfree_lift(eng::PFEngine)
    x, y, z = eng.xs
    R = eng.R
    K = eng.K

    # Only use the x- and z-Jacobian generators in the y-free sector
    fx = gens(eng.J)[1]
    fz = gens(eng.J)[3]

    # Cache the linear system data degree-by-degree
    cache = Dict{Int, Any}()

    function lift(h, J)
        # If y appears, fall back to the generic homogeneous lift
        if !is_yfree(h, 2)
            return lift_jacobian_homogeneous(h, J; weights=eng.weights)
        end

        D = weighted_degree_homogeneous(h, eng.weights)

        if !haskey(cache, D)
            # In the y-free sector with wt(x)=wt(z)=1, weighted degree D is just
            # ordinary homogeneous degree D in (x,z).
            target_basis = [x^a * z^(D-a) for a in 0:D]

            # Since fx and fz have degree 3, the quotient degree is D-3
            qdeg = D - 3
            qdeg < 0 && error("Need D >= 3 for a y-free lift into (fx,fz).")

            qbasis = [x^a * z^(qdeg-a) for a in 0:qdeg]

            # Unknown lift:
            #   h = q1*fx + q3*fz
            cols = vcat([m * fx for m in qbasis],
                        [m * fz for m in qbasis])

            # Build the linear system in the target monomial basis
            Adata = [coeff(cols[j], target_basis[i])
                     for i in 1:length(target_basis), j in 1:length(cols)]

            A = matrix(K, Adata)

            cache[D] = (target_basis=target_basis, qbasis=qbasis, A=A)
        end

        data = cache[D]
        rhs = [coeff(h, m) for m in data.target_basis]

        ok, sol = can_solve_with_solution(data.A, rhs; side=:right)
        ok || error("Could not solve y-free homogeneous lift system in degree $D.")

        v = unpack_solution(sol)
        nq = length(data.qbasis)

        # Reconstruct the quotient polynomials q1 and q3
        q1 = zero(R)
        q3 = zero(R)

        for i in 1:nq
            q1 += v[i] * data.qbasis[i]
            q3 += v[nq + i] * data.qbasis[i]
        end

        # No y-generator contribution is needed in the y-free sector
        q2 = zero(R)
        return [q1, q2, q3]
    end

    return lift
end


# Build a faster y-free lift with a gauge choice that makes the linear system square.
#
# As above, this is valid only in the weighted model where ∂_y f = y.
# The idea is:
#   h = q1*fx + q3*fz
# but instead of taking both q1 and q3 fully general, we restrict q3 to have
# x-degree <= 2.  This fixes the nonuniqueness of the lift and turns the solve
# into a square system, which is much faster symbolically.
#
# This is the preferred lift for the weighted symmetric-slice engine.
function make_square_yfree_lift(eng::PFEngine)
    x, y, z = eng.xs
    R = eng.R
    K = eng.K

    fx = gens(eng.J)[1]
    fz = gens(eng.J)[3]

    # Cache the degree-D linear systems
    cache = Dict{Int, Any}()

    function lift(h, J)
        # If y appears, fall back to the generic homogeneous lift
        if !is_yfree(h, 2)
            return lift_jacobian_homogeneous(h, J; weights=eng.weights)
        end

        D = weighted_degree_homogeneous(h, eng.weights)
        qdeg = D - 3
        qdeg < 0 && error("Need D >= 3 for a y-free lift.")

        if !haskey(cache, D)
            # Degree-D target monomials in the y-free sector
            target_basis = [x^a * z^(D-a) for a in 0:D]

            # q1 is taken fully general in degree qdeg
            q1_basis = [x^a * z^(qdeg-a) for a in 0:qdeg]

            # q3 is gauge-fixed to x-degree <= 2
            q3_basis = [x^a * z^(qdeg-a) for a in 0:min(2, qdeg)]

            cols = vcat([m * fx for m in q1_basis],
                        [m * fz for m in q3_basis])

            # Square linear system for the lift coefficients
            Adata = [coeff(cols[j], target_basis[i])
                     for i in 1:length(target_basis), j in 1:length(cols)]

            A = matrix(K, Adata)

            cache[D] = (
                target_basis = target_basis,
                q1_basis = q1_basis,
                q3_basis = q3_basis,
                A = A
            )
        end

        data = cache[D]
        rhs = [coeff(h, m) for m in data.target_basis]

        ok, sol = can_solve_with_solution(data.A, rhs; side = :right)
        ok || error("Could not solve square y-free lift system in degree $D.")

        v = unpack_solution(sol)

        # Reconstruct q1
        q1 = zero(R)
        for i in 1:length(data.q1_basis)
            q1 += v[i] * data.q1_basis[i]
        end

        # Reconstruct q3 from the remaining entries
        q3 = zero(R)
        off = length(data.q1_basis)
        for i in 1:length(data.q3_basis)
            q3 += v[off + i] * data.q3_basis[i]
        end

        # Again, no y-generator piece is needed
        q2 = zero(R)

        # Sanity check: verify the reconstructed lift exactly matches h
        check = q1 * fx + q3 * fz
        check == h || error("Square y-free lift reconstruction failed in degree $D.")

        return [q1, q2, q3]
    end

    return lift
end

In [ ]:
# make the square lift function for the symmetric slice
eng.lift_fn = make_yfree_lift(eng);

In [ ]:
# make the square 
eng.lift_fn = make_square_yfree_lift(eng);

In [ ]:
@time debug_derivative_class_compact(eng, 2, :E)

As we see this brute force approach hangs up due to the reduction of the second derivatives. So we try a different approach with a basis adapted to the periods we're interested in 

# Period adapted cohomology basis
Here we reduce the numerator that really matter to get 
$$\partial_E \sigma = \frac{\Omega}{f}$$ and $$\partial_E^2 \sigma = \frac{(z^4-x^2 z^2)\Omega}{f}$$
and define the pole-cleaned combintation 
$$ \Sigma = \sigma+ 2\mu_0 \partial_{\mu_0} \sigma + 2\mu_1 \partial_{\mu_2} \sigma + \mu_2 \partial_{\mu_2}\sigma $$
We reduce both $\Sigma$ and $\partial^2_E \sigma$ in the basis  in the basis $\{ \Omega/f, z^4\Omega/f^2\}$

From the affine reduction we found that 
$\Sigma = 2(E-1) + 2x^2 - 3 \mu_2 x) \Omega /f$. The global degree 4 piece corresponding to the nonconstant part is $P_A = 2x^2z^2-3\mu_2 x z^3$ and $P_B = z^4 - x^2 z^2$. So we want to reduce $P_A$, reduce $P_B$, read off the coefficients and solve for $a$ and $b$ in $\partial^2_E\sigma=a\partial_E \sigma +b\Sigma$. Then after integrating we get 
$$ N''_{0,\gamma} = a N'_{0,\gamma} + b\left(1 + 2\mu_0 \partial_{\mu_0} + 2 \mu_1\partial_{\mu_1}+\mu_2 \partial_{\mu_2} \right) N_{0,\gamma}$$



In [ ]:
# Compute the weighted-basis coordinates of
#   ∂_E^2 σ = ((z^4 - x^2 z^2) Ω) / f^2.
#
# The numerator P = z^4 - x^2 z^2 is Griffiths-Dwork reduced in the weighted
# P(1,2,1) model and then projected onto the basis
#   {1, z^4}  <->  {Ω/f, z^4 Ω/f^2}.
#
# The output is a named tuple giving the coefficients (c0, c4), i.e.
#   c0 * Ω/f + c4 * z^4 Ω/f^2.
function coords_of_dE2sigma(eng)
    x, y, z = eng.xs
    P = z^4 - x^2*z^2
    red = pole_reduce_full(P, eng.f, eng.J; lift_fn=eng.lift_fn, weights=eng.weights)
    return reduce_to_basis(eng, red)
end

In [ ]:
# NOTE:
# This was a first brute-force attempt to extract the classical PF data entirely
# inside the weighted P(1,2,1) model.
#
# The B-piece is correct in this formulation:
#   d_E^2 sigma = ((z^4 - x^2 z^2) Ω)/f^2
# is naturally represented by the weighted degree-4 numerator
#   P_B = z^4 - x^2 z^2,
# which can be reduced directly into the basis
#   {Ω/f, z^4 Ω/f^2}.
#
# The A-piece, however, is NOT correctly captured by the weighted numerator
#   P_A = 2 x^2 z^2 - 3 mu2 x z^3.
# That guess was too naive.
#
# What I learned later is that the correct A-reduction has to be done in the
# unweighted P^2 model, by reducing the numerator
#   2 x^2 y^2 zeta
# against the unweighted Jacobian ideal and then decomposing the result in the
# span {zeta^5, x, zeta}.  Only after that can one identify the coefficients
# corresponding to z^4 Ω/f^2 and Ω/f.
#
# So:
#   - coords_of_dE2sigma below is fine,
#   - coords_of_sigma_combo below is *not* the final correct A-step,
#     and the resulting (a,b) do not match the validated PF data.
#
# We keep this code as a record of the initial weighted-only attempt.


# Reduce a weighted-degree-4 numerator P to
# c0 * Ω/f + c4 * z^4 Ω/f^2
function reduce_deg4_class(eng::PFEngine, P)
    red = pole_reduce_full(P, eng.f, eng.J; lift_fn=eng.lift_fn, weights=eng.weights)
    return reduce_to_basis(eng, red)
end

# B from d_E^2 sigma = ((z^4 - x^2 z^2) Ω)/f^2
function coords_of_dE2sigma(eng::PFEngine)
    x, y, z = eng.xs
    P_B = z^4 - x^2*z^2
    B = reduce_deg4_class(eng, P_B)
    return (P=P_B, c0=B.c0, c4=B.c4)
end

# A from the extra part of
# Sigma = sigma + 2mu0 d_mu0 sigma + 2mu1 d_mu1 sigma + mu2 d_mu2 sigma
#       = 2(E-1) Ω/f + [extra degree-4 class]
function coords_of_sigma_combo(eng::PFEngine)
    x, y, z = eng.xs
    mu2 = eng.params[:mu2]
    E   = eng.params[:E]

    P_A = 2*x^2*z^2 - 3*mu2*x*z^3
    A = reduce_deg4_class(eng, P_A)

    return (
        P = P_A,
        extra_c0 = A.c0,
        extra_c4 = A.c4,
        total_c0 = 2*(E - 1) + A.c0,
        total_c4 = A.c4
    )
end

# Solve for the PF data
# d_E^2 sigma = a * d_E sigma + b * Sigma
#
# where d_E sigma = Ω/f,
# Sigma = total_c0 * Ω/f + total_c4 * z^4 Ω/f^2,
# d_E^2 sigma = B.c0 * Ω/f + B.c4 * z^4 Ω/f^2
function pf_ab_from_A_B(eng::PFEngine)
    A = coords_of_sigma_combo(eng)
    B = coords_of_dE2sigma(eng)

    iszero(A.total_c4) && error("Sigma c4 coefficient vanished; b would be singular on this locus.")

    b = B.c4 / A.total_c4
    a = B.c0 - b * A.total_c0

    return (A=A, B=B, a=a, b=b)
end

In [ ]:
@time A = coords_of_sigma_combo(eng)
println("A numerator = ", A.P)
println("A extra coefficients = ", (A.extra_c0, A.extra_c4))
println("A total coefficients = ", (A.total_c0, A.total_c4))

In [ ]:
@time B = coords_of_dE2sigma(eng)
println("B numerator = ", B.P)
println("B coefficients = ", (B.c0, B.c4))

In [ ]:
@time out = pf_ab_from_A_B(eng)
println("a = ", out.a)
println("b = ", out.b)

In [ ]:
# helper function to reduce to the symmetric case
function specialize_coeff(a, vars, vals)
    num2 = evaluate(numerator(a), vars, vals)
    den2 = evaluate(denominator(a), vars, vals)
    return num2 // den2
end


# symmetric case mu1=mu2=0 
vars = [eng.params[:mu1], eng.params[:mu2]];
vals = [ 0, 0];

# pick some random numbers to check  
varsAll = [eng.params[:E], eng.params[:mu0],eng.params[:mu1], eng.params[:mu2]];
vals2 = [0, 3, 1, 2];


In [ ]:
a_00  = specialize_coeff(out.a,     vars, vals);
b_00  = specialize_coeff(out.b,     vars, vals);


println("a |μ1=μ2=0 = ", a_00)
println("b |μ1=μ2=0 = ", b_00)


Note that this isn't the same system as in Max's SAGE calculation. Above, the A-piece was being modeled too naively. I tried to encode the extra part of 
$\Sigma$ directly as a weighted degree-4 numerator $P_A = 2x^2 z^2 - 3\mu_2 xz^3$ and then reduce it straight into the two-dimensional weighted basis. But Max we reducing in the unweighted $\mathbb P^2$ model with $2 x^2 y^2 \zeta$ against the unweighted Jacobian ideal in the span $\{\zeta^5,x,\zeta\}$

In [ ]:
# NOTE:
# This was the second weighted-only attempt to extract the classical PF data.
#
# Compared to the earlier brute-force version, this one uses the cleaner
# projective representative
#   P_A = 2 x^2 z^2
# for the extra piece in Sigma, following the idea that
#   Sigma = sigma + 2mu0 d_mu0 sigma + 2mu1 d_mu1 sigma + 2mu2 d_mu2 sigma
#         = 2(E-1) Ω/f + [2 x^2 z^2 Ω / f^2]
# in projective form.
#
# This is better than the older weighted ansatz, but it still does NOT give the
# final validated A-data.  The reason is that the true A-reduction must be carried
# out in the unweighted P^2 model, where one reduces
#   2 x^2 y^2 zeta
# against the unweighted Jacobian ideal and decomposes the result in
#   {zeta^5, x, zeta}.
#
# So:
#   - the B-piece below is correct as a weighted reduction,
#   - the linear solve for (a,b) is algebraically correct once A and B are known,
#   - but the weighted-only A-piece below is still not the final correct one.
#
# We keep this code as a record of the intermediate "projectively improved"
# weighted approach.


# reduce a weighted-degree-4 numerator P and return coefficients in
#   c1 * Ω/f + c4 * z^4 Ω/f^2
function reduce_deg4_class(eng::PFEngine, P)
    red = pole_reduce_full(P, eng.f, eng.J; lift_fn=eng.lift_fn, weights=eng.weights)
    return reduce_to_basis(eng, red)
end

# Max's A-data:
# Sigma = sigma + 2mu0 d_mu0 sigma + 2mu1 d_mu1 sigma + 2mu2 d_mu2 sigma
#       = 2(E-1) Ω/f + [2 x^2 z^2 Ω / f^2]  in projective representation
function coords_of_sigma_combo_max(eng::PFEngine)
    x, y, z = eng.xs
    E = eng.params[:E]

    P_A = 2 * x^2 * z^2
    A = reduce_deg4_class(eng, P_A)

    return (
        P = P_A,
        A1 = A.c0,                  # coeff of Ω/f from reducing P_A
        A0 = A.c4,                  # coeff of z^4 Ω/f^2 from reducing P_A
        total1 = 2*(E - 1) + A.c0,  # total coeff of Ω/f in Sigma
        total0 = A.c4               # total coeff of z^4 Ω/f^2 in Sigma
    )
end

# B-data:
# d_E^2 sigma = (z^4 - x^2 z^2) Ω / f^2
function coords_of_dE2sigma_max(eng::PFEngine)
    x, y, z = eng.xs

    P_B = z^4 - x^2 * z^2
    B = reduce_deg4_class(eng, P_B)

    return (
        P = P_B,
        B1 = B.c0,   # coeff of Ω/f
        B0 = B.c4    # coeff of z^4 Ω/f^2
    )
end

function pf_ab_max(eng::PFEngine)
    A = coords_of_sigma_combo_max(eng)
    B = coords_of_dE2sigma_max(eng)

    iszero(A.total0) && error("A0 vanished, so b = B0/A0 is singular on this locus.")

    b = B.B0 / A.total0
    a = B.B1 - b * A.total1

    return (A=A, B=B, a=a, b=b)
end

In [ ]:
@time A = coords_of_sigma_combo_max(eng)
println("P_A = ", A.P)
println("A1  = ", A.A1)
println("A0  = ", A.A0)
println("total coeffs in Sigma = ", (A.total1, A.total0))

In [ ]:
x, y, z = eng.xs
println("Sigma extra numerator should be: ", 2*x^2*z^2)
println("dE^2 sigma numerator should be: ", z^4 - x^2*z^2)

In [ ]:
out = pf_ab_max(eng)

In [ ]:
A1_00 = specialize_coeff(out.A.A1, vars, vals)
A0_00 = specialize_coeff(out.A.A0, vars, vals)
B1_00 = specialize_coeff(out.B.B1, vars, vals)
B0_00 = specialize_coeff(out.B.B0, vars, vals)
a_00  = specialize_coeff(out.a,     vars, vals)
b_00  = specialize_coeff(out.b,     vars, vals)

println("A1|μ1=μ2=0 = ", A1_00)
println("A0|μ1=μ2=0 = ", A0_00)
println("B1|μ1=μ2=0 = ", B1_00)
println("B0|μ1=μ2=0 = ", B0_00)
println("a |μ1=μ2=0 = ", a_00)
println("b |μ1=μ2=0 = ", b_00)

After residues 
$$ 1 = \mathrm{Res} \frac{\Omega}{f} \ \ \ z^4 = \mathrm{Res}{z^4 \Omega}{f^2} $$
The identity 
$$ \Sigma = \sigma + 2 \mu_i \partial_i \sigma  = 2(E-1) \frac{\Omega}{f} + 2 \frac{x^2}{z^2}\frac{\Omega}{f}$$
where $i=0..2$. Representing the last term projectively $2x^2z^2 = \mathrm{Res}\frac{2x^2z^2\Omega}{f}$ gives
the specialization $$ \Sigma = \left(2(E-1)+A_1)\right)\frac{\Omega}{f} + \frac{A_0 z^4\Omega}{f}$$ where $A_0$ and $A_1$ come from reducing down $2x^2 z^2$ and $\partial_E^2\sigma = B_1 \frac{\Omega}{f} + B_0 \frac{z^4\Omega}{f^2} = \frac{(z^4-x^2z^2)\Omega}{f}$. 

Then we solve for $\partial_E^2\sigma = a\partial_E \sigma + b\Sigma$ as before with $\partial_E\sigma=\Omega/f$
so the classical PF data $$b=B_0/A_0$$ and $$a=B_1-b(2(E-1)+A_1)$$. 

The correction relative to the previous attempt is the $\mu_2$ coefficient is $2\mu_2$ not $\mu_2$ and the extra 
numerator for $\Sigma$ is just $2(xz)^2$ not something with the $xz^3$ term. 

In [ ]:
# Reduce a weighted degree-4 numerator P and decompose the result in the
# three directions
#   {1, z^4, x z^3}
# corresponding to
#   {Ω/f, z^4 Ω/f^2, extra x-direction}.
#
# This was the key improvement over the earlier weighted-only attempts:
# instead of forcing the answer directly into the 2d basis {Ω/f, z^4 Ω/f^2},
# we allow the additional degree-4 direction x z^3, which captures the
# mu2 / x contribution that naturally appears in the A-step.
function reduce_to_three_directions(eng::PFEngine, P)
    x, y, z = eng.xs

    # Griffiths-Dwork reduction in the weighted model
    red = pole_reduce_full(P, eng.f, eng.J; lift_fn=eng.lift_fn, weights=eng.weights)
    r = normal_form(red, eng.J)

    # Only degree 0 and degree 4 pieces are relevant here
    p0 = homogeneous_piece_of_degree(r, eng.weights, 0)
    p4 = homogeneous_piece_of_degree(r, eng.weights, 4)

    # Degree-0 piece is automatically a multiple of 1
    c1 = iszero(p0) ? zero(eng.K) : coeff(p0, one(eng.R))

    # Degree-4 piece is decomposed in the basis {z^4, x z^3}
    if iszero(p4)
        c4 = zero(eng.K)
        cx = zero(eng.K)
    else
        v = quotient_degree_coords(eng, p4, 4, [z^4, x*z^3])
        length(v) == 2 || error("Expected 2 coordinates in degree 4, got $v")
        c4, cx = v
    end

    # Return:
    #   c1 = coeff of Ω/f,
    #   c4 = coeff of z^4 Ω/f^2,
    #   cx = coeff of the extra x z^3 direction.
    return (c1=c1, c4=c4, cx=cx, reduced=r)
end


# Extract the weighted A-data from the "difficult piece"
#   2 x^2 z^2.
#
# This is the stage where the weighted computation starts behaving properly:
# the reduced answer is allowed to have an extra x z^3 component, recorded in Ax,
# instead of being forced too early into the 2d basis.
#
# In the symmetric slice this extra direction later vanishes, but allowing it here
# was the key structural fix.
function A_data(eng::PFEngine)
    x, y, z = eng.xs
    E = eng.params[:E]
    mu2 = eng.params[:mu2]

    # Reduce the difficult piece in the 3-direction basis
    
    # Reduce a class and decompose it into
    #   c1 * Ω/f  +  c4 * z^4 Ω/f^2  +  cx * (xz^3)-direction
    #
    # The point is to reduce the difficult piece, then separate off the mu2-direction
    Araw = reduce_to_three_directions(eng, 2*x^2*z^2)

    return (
        raw = Araw,
        A1 = Araw.c1,              # coeff of Ω/f
        A0 = Araw.c4,              # coeff of z^4 Ω/f^2
        Ax = Araw.cx,              # coeff of the extra x z^3 direction
        total1 = 2*(E - 1) + Araw.c1
    )
end


# Extract the weighted B-data from
#   d_E^2 sigma = (z^4 - x^2 z^2) Ω / f^2.
#
# Unlike the A-step, the B-step already naturally lands in the 2d cohomology basis,
# but we still record the possible extra x z^3 component as Bx for diagnostics.
function B_data(eng::PFEngine)
    x, y, z = eng.xs

    Braw = reduce_to_three_directions(eng, z^4 - x^2*z^2)

    return (
        raw = Braw,
        B1 = Braw.c1,              # coeff of Ω/f
        B0 = Braw.c4,              # coeff of z^4 Ω/f^2
        Bx = Braw.cx               # coeff of the extra x z^3 direction
    )
end


# Solve for the classical PF coefficients (a,b) from the 3-direction reduction.
#
# This uses only the Ω/f and z^4 Ω/f^2 components:
#   d_E^2 sigma = a * d_E sigma + b * Sigma,
# with
#   d_E sigma = Ω/f.
#
# The extra x-direction is carried along in A.Ax and B.Bx for diagnostics,
# but does not enter this solve directly.
#
# Historically, this was the first weighted setup that started to reveal the
# correct structure, because it no longer hid the extra mu2-direction.
function ab_from_three_direction_reduction(eng::PFEngine)
    A = A_data(eng)
    B = B_data(eng)

    iszero(A.A0) && error("A0 vanished, cannot solve for b.")

    b = B.B0 / A.A0
    a = B.B1 - b * (2*(eng.params[:E] - 1) + A.A1)

    return (A=A, B=B, a=a, b=b)
end

In [ ]:
A = A_data(eng)
B = B_data(eng)

println("A1 = ", A.A1)
println("A0 = ", A.A0)
println("Ax = ", A.Ax)
println()
println("B1 = ", B.B1)
println("B0 = ", B.B0)
println("Bx = ", B.Bx)

In [ ]:
out3 = ab_from_three_direction_reduction(eng)
println("a = ", out3.a)
println("b = ", out3.b)


a_val = specialize_coeff(out3.a,     varsAll, vals2);
b_val  = specialize_coeff(out3.b,     varsAll , vals2);
println("aval = ", a_val)
println("bval = ", b_val)

In [ ]:
println("a = ", specialize_coeff(out3.a, vars, ))

In [ ]:
vars = [eng.params[:mu1], eng.params[:mu2]]
vals = [0, 0]

println("A1|00 = ", specialize_coeff(out3.A.A1, vars, vals))
println("A0|00 = ", specialize_coeff(out3.A.A0, vars, vals))
println("Ax|00 = ", specialize_coeff(out3.A.Ax, vars, vals))
println("B1|00 = ", specialize_coeff(out3.B.B1, vars, vals))
println("B0|00 = ", specialize_coeff(out3.B.B0, vars, vals))
println("Bx|00 = ", specialize_coeff(out3.B.Bx, vars, vals))
println("a |00 = ", specialize_coeff(out3.a, vars, vals))
println("b |00 = ", specialize_coeff(out3.b, vars, vals))

In [ ]:
# Build the auxiliary unweighted P^2 model used for the "A-piece" reduction.
#
# This uses the SAME coefficient field K = Frac(QQ[mu0,mu1,mu2,E]) as the
# weighted engine, but changes the ambient ring from P(1,2,1) to ordinary P^2
# with homogeneous coordinates (x,y,zeta), all of degree 1.
#
# Geometrically, this is the model in which the difficult piece
#   2 x^2 y^2 zeta
# is reduced against the Jacobian ideal of f_un, and then decomposed in the
# basis {zeta^5, x, zeta}.  This is the setup that matches Max's
# SAGE notebook for the A-reduction: 
# S_un.<x,y,zeta> = PolynomialRing(F, order=TermOrder('wdegrevlex', (1,1,1)))
# f_un =  1/2*y^2*zeta^2 + x^4  + (-mu_2)*x^3*zeta + (E - 2)*x^2*zeta^2 
# + (mu_1 + mu_2)*x*zeta^3 + (mu_0 - E + 1)*zeta^4
# J_un = S_un.ideal([diff(f_un,x),diff(f_un,y),diff(f_un,zeta)])
#
function make_unweighted_engine_from(eng::PFEngine)
    # Reuse the same coefficient field and parameter symbols from the weighted engine
    K = eng.K
    params = eng.params

    mu0 = params[:mu0]
    mu1 = params[:mu1]
    mu2 = params[:mu2]
    E   = params[:E]

    # Ordinary projective plane P^2: all variables have weight 1
    R, (x, y, zeta) = graded_polynomial_ring(K, [:x, :y, :zeta], [1,1,1])

    # Unweighted homogeneous quartic defining the same family in P^2 coordinates
    f = K(1//2)*y^2*zeta^2 +
        x^4 -
        mu2*x^3*zeta +
        (E - K(2))*x^2*zeta^2 +
        (mu1 + mu2)*x*zeta^3 +
        (mu0 - E + K(1))*zeta^4

    # Jacobian ideal for Griffiths-Dwork reduction in the unweighted model
    J = ideal(R, [derivative(f, x), derivative(f, y), derivative(f, zeta)])

    # Construct a PFEngine container.
    #
    # The "basis_numerators" and "basis_pole_orders" entries are only placeholders
    # here: in the unweighted model we are not using the generic PF basis machinery
    # directly, but rather reducing specific classes and then identifying
    #   zeta^5 <-> z^4 Ω / f^2,
    #   zeta   <-> Ω / f,
    # with the x-direction treated separately when needed.
    eng_un = PFEngine(
        eng.T, K, R, [x,y,zeta], [1,1,1], f, J,
        params,
        Dict{Symbol, Any}(),     # no dfparams needed for this auxiliary engine
        [one(R), zeta^5],        # placeholder basis representatives
        [1, 2],                  # placeholder pole orders
        (h, J) -> error("lift_fn not installed yet")
    )

    # IMPORTANT:
    # Do NOT use the y-free lift optimization here.
    #
    # In the weighted P(1,2,1) model we had ∂_y f = y, so y lay in the Jacobian ideal
    # and many reductions could be restricted to the y-free sector.
    #
    # In the unweighted P^2 model, however,
    #   ∂_y f = y*zeta^2,
    # so y itself is NOT a Jacobian generator.  Therefore we must use the full
    # homogeneous lift in the variables (x,y,zeta).
    eng_un.lift_fn = (h, J) -> lift_jacobian_homogeneous(h, J; weights=[1,1,1])

    return eng_un
end


# Build the unweighted P^2 model specialized to the symmetric slice mu1 = mu2 = 0.
#
# This is the fast "sanity check" engine used to validate the classical PF data
# on the symmetric slice before attempting the full symbolic (mu0,mu1,mu2,E) run.
#
# Here the coefficient field is only Frac(QQ[mu0,E]), so the reductions are much
# cheaper and the resulting formulas can be compared directly with the known
# symmetric PF equation.
function make_unweighted_engine_mu1mu2_zero()
    # Coefficient field for the symmetric slice: only mu0 and E remain symbolic
    T0, (mu0, E) = polynomial_ring(QQ, [:mu0, :E])
    K0 = fraction_field(T0)

    # Ordinary projective plane P^2 with homogeneous coordinates (x,y,zeta)
    R0, (x, y, zeta) = graded_polynomial_ring(K0, [:x, :y, :zeta], [1,1,1])

    # Quartic specialized to mu1 = mu2 = 0
    f0 = K0(1//2)*y^2*zeta^2 +
         x^4 +
         (E - K0(2))*x^2*zeta^2 +
         (mu0 - E + K0(1))*zeta^4

    # Jacobian ideal on the symmetric slice
    J0 = ideal(R0, [derivative(f0, x), derivative(f0, y), derivative(f0, zeta)])

    # Only the surviving parameters are stored in the slice engine
    params0 = Dict(
        :mu0 => mu0,
        :E   => E,
    )

    # As above, the basis data are placeholders: the actual use of this engine is
    # to reduce the A-piece in the unweighted model and then read off the
    # coefficients of zeta^5, x, and zeta.
    eng0 = PFEngine(
        T0, K0, R0, [x,y,zeta], [1,1,1], f0, J0,
        params0,
        Dict{Symbol,Any}(),   # no dfparams needed here
        [one(R0), zeta^5],    # placeholder basis representatives
        [1,2],                # placeholder pole orders
        (h, J) -> error("lift_fn not installed yet")
    )

    # Again, use the full homogeneous lift in the unweighted variables.
    eng0.lift_fn = (h, J) -> lift_jacobian_homogeneous(h, J; weights=[1,1,1])

    return eng0
end


# Build the weighted P(1,2,1) engine specialized to the symmetric slice mu1 = mu2 = 0.
#
# This is the fast slice engine used for the weighted B-reduction:
#   (z^4 - x^2 z^2) Ω / f^2.
#
# Here the y-free square lift is valid again, since in the weighted model
# one has ∂_y f = y.
function make_weighted_engine_mu1mu2_zero()
    # Coefficient field on the symmetric slice
    T0, (mu0, E) = polynomial_ring(QQ, [:mu0, :E])
    K0 = fraction_field(T0)

    # Weighted projective model P(1,2,1)
    R0, (x, y, z) = graded_polynomial_ring(K0, [:x, :y, :z], [1,2,1])

    # Quartic specialized to mu1 = mu2 = 0
    f0 = K0(1//2)*y^2 +
         x^4 +
         (E - K0(2))*x^2*z^2 +
         (mu0 - E + K0(1))*z^4

    # Jacobian ideal for weighted reduction
    J0 = ideal(R0, [derivative(f0, x), derivative(f0, y), derivative(f0, z)])

    params0 = Dict(
        :mu0 => mu0,
        :E   => E,
    )

    # Standard weighted engine with basis {1, z^4}
    eng0 = PFEngine(
        T0, K0, R0, [x,y,z], [1,2,1], f0, J0,
        params0,
        Dict{Symbol,Any}(),  # no dfparams needed for the slice B-step
        [one(R0), z^4],
        [1,2],
        (h, J) -> error("lift_fn not installed yet")
    )

    # In the weighted model we can use the optimized square y-free lift
    eng0.lift_fn = make_square_yfree_lift(eng0)
    return eng0
end

In [ ]:
# Reduce an unweighted P^2 numerator P and decompose the result in the span
#   {zeta^5, x, zeta}.
#
# Interpretation:
#   zeta^5  <->  z^4 Ω / f^2,
#   zeta    <->  Ω / f,
#   x       <->  the extra mu2-direction.
#
# If subtract_mu2x=true, subtract the expected mu2*x piece before projecting.
function decompose_unweighted_A_piece(eng_un::PFEngine, P; subtract_mu2x=false)
    x, y, zeta = eng_un.xs

    # Griffiths-Dwork reduction in the unweighted model
    red = pole_reduce_full(P, eng_un.f, eng_un.J;
                           lift_fn=eng_un.lift_fn,
                           weights=eng_un.weights)

    # Optionally remove the explicit mu2*x contribution
    if subtract_mu2x
        red -= eng_un.params[:mu2] * x
    end

    # Pass to a normal form representative modulo the Jacobian ideal
    r = normal_form(red, eng_un.J)

    # Only degrees 1 and 5 should survive in the final answer
    p1 = homogeneous_piece_of_degree(r, eng_un.weights, 1)
    p5 = homogeneous_piece_of_degree(r, eng_un.weights, 5)

    # Check that no unexpected homogeneous pieces remain
    extra = zero(eng_un.R)
    for comp in values(homogeneous_components(r))
        if !iszero(comp)
            d = weighted_degree_homogeneous(comp, eng_un.weights)
            if d != 1 && d != 5
                extra += comp
            end
        end
    end
    iszero(extra) || error("Unexpected degrees in unweighted reduction: $extra")

    # Degree-5 piece is 1-dimensional: project onto zeta^5
    c5 = iszero(p5) ? zero(eng_un.K) :
         quotient_degree_coords(eng_un, p5, 5, [zeta^5])[1]

    # Degree-1 piece is 2-dimensional: decompose in {x, zeta}
    if iszero(p1)
        cx = zero(eng_un.K)
        c1 = zero(eng_un.K)
    else
        v = quotient_degree_coords(eng_un, p1, 1, [x, zeta])
        length(v) == 2 || error("Expected 2 coords in degree 1, got $v")
        cx, c1 = v
    end

    # Return:
    #   c5 = coeff of zeta^5,
    #   cx = coeff of x,
    #   c1 = coeff of zeta.
    return (c5=c5, cx=cx, c1=c1, reduced=r)
end



In [ ]:
# Unweighted A-reduction on the symmetric slice.
# Decomposes 2 x^2 y^2 zeta into {zeta^5, x, zeta} and returns
# the coefficients (A0, Ax, A1) = (c5, cx, c1).
function A_unweighted_slice(eng_un0::PFEngine)
    x, y, zeta = eng_un0.xs
    P = 2*x^2*y^2*zeta
    raw = decompose_unweighted_A_piece(eng_un0, P; subtract_mu2x=false)
    return (P=P, raw=raw, A0=raw.c5, A1=raw.c1, Ax=raw.cx)
end

In [ ]:
eng_un0 = make_unweighted_engine_mu1mu2_zero()
Aun0 = A_unweighted_slice(eng_un0)

println("raw A decomposition   = ", (Aun0.raw.c5, Aun0.raw.cx, Aun0.raw.c1))
println("A0 = ", Aun0.A0)
println("A1 = ", Aun0.A1)
println("Ax = ", Aun0.Ax)

In [ ]:
# Compute the unweighted A-data in the auxiliary P^2 model.
#
# This is Max's A-step:
# reduce the numerator
#   P = 2 x^2 y^2 zeta
# in the unweighted model, where the natural decomposition is in the span
#   {zeta^5, x, zeta}.
#
# We compute two versions:
#   raw   = direct reduction of P,
#   clean = reduction after subtracting the expected mu2*x piece.
#
# Interpretation:
#   A0 = coeff of zeta^5,
#   A1 = coeff of zeta,
#   Ax = coeff of x after the mu2*x subtraction.
#
# In the fully correct setup, Ax should vanish after the subtraction.
function A_unweighted(eng_un::PFEngine)
    x, y, zeta = eng_un.xs
    # One small code note: since mu2 is not actually used in the body here anymore, you could delete this line
    mu2 = get(eng_un.params, :mu2, zero(eng_un.K))

    # Numerator of the "difficult piece" in the unweighted model
    P = 2 * x^2 * y^2 * zeta

    # Raw reduction and cleaned reduction (with the mu2*x piece removed)
    raw   = decompose_unweighted_A_piece(eng_un, P; subtract_mu2x=false)
    clean = decompose_unweighted_A_piece(eng_un, P; subtract_mu2x=true)

    # Return both versions for diagnostics, and the cleaned coefficients
    return (
        P = P,
        raw = raw,
        clean = clean,
        A0 = clean.c5,
        A1 = clean.c1,
        Ax = clean.cx
    )
end

# Compute the weighted B-data from
#   d_E^2 sigma = (z^4 - x^2 z^2) Ω / f^2.
#
# This is the clean part of the classical PF calculation:
# the numerator
#   P = z^4 - x^2 z^2
# is genuinely a weighted degree-4 class in the P(1,2,1) model, so it can be
# reduced directly into the weighted cohomology basis
#   {1, z^4}  <->  {Ω/f, z^4 Ω/f^2}.
#
# The output coefficients are:
#   B1 = coeff of Ω/f,
#   B0 = coeff of z^4 Ω/f^2.
function B_weighted(eng::PFEngine)
    x, y, z = eng.xs
    P = z^4 - x^2*z^2
    B = reduce_deg4_class(eng, P)
    return (P=P, B1=B.c0, B0=B.c4)
end

In [ ]:
eng_w0 = make_weighted_engine_mu1mu2_zero()

@time B0 = B_weighted(eng_w0)
println("B0 = ", B0.B0)
println("B1 = ", B0.B1)

In [ ]:
# Compute the classical PF coefficients (a,b) using the mixed setup:
#
#   - A is computed in the unweighted P^2 model, where the difficult piece
#       2 x^2 y^2 zeta
#     is reduced and decomposed in {zeta^5, x, zeta};
#
#   - B is computed in the weighted P(1,2,1) model, where
#       d_E^2 sigma = (z^4 - x^2 z^2) Ω / f^2
#     is naturally reduced in the basis {Ω/f, z^4 Ω/f^2}.
#
# This is the validated classical construction:
# the earlier weighted-only attempts got B right, but not A.
#
# The solve uses
#   d_E^2 sigma = a * d_E sigma + b * Sigma,
# with
#   d_E sigma = Ω/f
# and
#   Sigma = (2(E-1)+A1) Ω/f + A0 z^4 Ω/f^2.
#
# We keep a warning if the unweighted A-step still leaves an x-piece after the
# mu2*x subtraction, since that would indicate the reduction did not fully clean
# the extra direction.
function ab_mixed(eng::PFEngine, eng_un::PFEngine)
    A = A_unweighted(eng_un)
    B = B_weighted(eng)

    iszero(A.Ax) || @warn "A still has residual x-piece after subtracting mu2*x" A.Ax
    iszero(A.A0) && error("A0 vanished; cannot solve for b.")

    b = B.B0 / A.A0
    a = B.B1 - b * (2*(eng.params[:E] - 1) + A.A1)

    return (A=A, B=B, a=a, b=b)
end

## symmetric slice decomposition

In [ ]:
# Unweighted A-reduction on the symmetric slice mu1 = mu2 = 0.
#
# Reduce the difficult numerator
#   P = 2 x^2 y^2 zeta
# in the unweighted P^2 model and decompose the result in
#   {zeta^5, x, zeta}.
#
# Since mu2 = 0 on this slice, no mu2*x subtraction is needed:
# the raw decomposition is already the relevant one.
#
# Output:
#   A0 = coeff of zeta^5,
#   A1 = coeff of zeta,
#   Ax = coeff of x.
function A_unweighted_slice(eng_un0::PFEngine)
    x, y, zeta = eng_un0.xs

    P = 2 * x^2 * y^2 * zeta
    raw = decompose_unweighted_A_piece(eng_un0, P; subtract_mu2x=false)

    return (
        P = P,
        raw = raw,
        A0 = raw.c5,
        A1 = raw.c1,
        Ax = raw.cx
    )
end

In [ ]:
# PF data on the slice 
function ab_slice(eng_w0::PFEngine, eng_un0::PFEngine)
    A = A_unweighted_slice(eng_un0)
    B = B_weighted(eng_w0)

    iszero(A.A0) && error("A0 vanished; cannot solve for b.")

    b = B.B0 / A.A0
    a = B.B1 - b * (2*(eng_w0.params[:E] - 1) + A.A1)

    return (A=A, B=B, a=a, b=b)
end

In [ ]:
# symmetric decomposition on the slice
eng_un0 = make_unweighted_engine_mu1mu2_zero()
Aun0 = A_unweighted_slice(eng_un0)

println("raw A decomposition   = ", (Aun0.raw.c5, Aun0.raw.cx, Aun0.raw.c1))
println("A0 = ", Aun0.A0)
println("A1 = ", Aun0.A1)
println("Ax = ", Aun0.Ax)

In [ ]:
eng_un = make_unweighted_engine_from(eng)

In [ ]:
@time out0 = ab_slice(eng_w0, eng_un0)
println("a = ", out0.a)
println("b = ", out0.b)

In [ ]:
println("A0 factored = ", factor(Aun0.A0))
println("A1 factored = ", factor(Aun0.A1))
println("B0 factored = ", factor(B0.B0))
println("B1 factored = ", factor(B0.B1))
println("a factored  = ", factor(out0.a))
println("b factored  = ", factor(out0.b))

In [ ]:
function transform_compare(r, Evar, mu0var)
    num = evaluate(numerator(r),   [Evar, mu0var], [1 - Evar, -mu0var])
    den = evaluate(denominator(r), [Evar, mu0var], [1 - Evar, -mu0var])
    return num // den
end

E   = eng_w0.params[:E]
mu0 = eng_w0.params[:mu0]

a_cmp = transform_compare(out0.a, E, mu0)
b_cmp = transform_compare(out0.b, E, mu0)

println("a transformed = ", factor(a_cmp))
println("b transformed = ", factor(b_cmp))

In [ ]:
eng_un = make_unweighted_engine_from(eng)
out = ab_mixed(eng, eng_un)

In [ ]:
# Convert integers / rationals to QQ elements
toQQ(x::Integer) = QQ(x)
toQQ(x::Rational) = QQ(numerator(x), denominator(x))

# Weighted engine at a fixed parameter point (mu0,mu1,mu2,E)
function make_weighted_engine_at(mu0v, mu1v, mu2v, Ev)
    mu0v = toQQ(mu0v)
    mu1v = toQQ(mu1v)
    mu2v = toQQ(mu2v)
    Ev   = toQQ(Ev)

    R, (x, y, z) = graded_polynomial_ring(QQ, [:x, :y, :z], [1,2,1])

    f = QQ(1//2)*y^2 +
        x^4 -
        mu2v*x^3*z +
        (Ev - QQ(2))*x^2*z^2 +
        (mu1v + mu2v)*x*z^3 +
        (mu0v - Ev + QQ(1))*z^4

    J = ideal(R, [derivative(f, x), derivative(f, y), derivative(f, z)])

    params = Dict(
        :mu0 => mu0v,
        :mu1 => mu1v,
        :mu2 => mu2v,
        :E   => Ev,
    )

    eng0 = PFEngine(
        nothing, QQ, R, [x,y,z], [1,2,1], f, J,
        params,
        Dict{Symbol,Any}(),
        [one(R), z^4],
        [1,2],
        (h, J) -> error("lift_fn not installed yet")
    )

    eng0.lift_fn = make_square_yfree_lift(eng0)
    return eng0
end

# Unweighted P^2 engine at the same fixed parameter point
function make_unweighted_engine_at(mu0v, mu1v, mu2v, Ev)
    mu0v = toQQ(mu0v)
    mu1v = toQQ(mu1v)
    mu2v = toQQ(mu2v)
    Ev   = toQQ(Ev)

    R, (x, y, zeta) = graded_polynomial_ring(QQ, [:x, :y, :zeta], [1,1,1])

    f = QQ(1//2)*y^2*zeta^2 +
        x^4 -
        mu2v*x^3*zeta +
        (Ev - QQ(2))*x^2*zeta^2 +
        (mu1v + mu2v)*x*zeta^3 +
        (mu0v - Ev + QQ(1))*zeta^4

    J = ideal(R, [derivative(f, x), derivative(f, y), derivative(f, zeta)])

    params = Dict(
        :mu0 => mu0v,
        :mu1 => mu1v,
        :mu2 => mu2v,
        :E   => Ev,
    )

    eng_un = PFEngine(
        nothing, QQ, R, [x,y,zeta], [1,1,1], f, J,
        params,
        Dict{Symbol,Any}(),
        [one(R), zeta^5],
        [1,2],
        (h, J) -> error("lift_fn not installed yet")
    )

    eng_un.lift_fn = (h, J) -> lift_jacobian_homogeneous(h, J; weights=[1,1,1])
    return eng_un
end

In [ ]:
# Example: replace these by the parameter values you used in Sage
mu0v = 3
mu1v = 1
mu2v = 2
Ev   = 4

eng_num    = make_weighted_engine_at(mu0v, mu1v, mu2v, Ev)
eng_un_num = make_unweighted_engine_at(mu0v, mu1v, mu2v, Ev)

@time out_num = ab_mixed(eng_num, eng_un_num)

println("A0 = ", out_num.A.A0)
println("A1 = ", out_num.A.A1)
println("Ax = ", out_num.A.Ax)
println("B0 = ", out_num.B.B0)
println("B1 = ", out_num.B.B1)
println("a  = ", out_num.a)
println("b  = ", out_num.b)

In [ ]:
a_val = specialize_coeff(out.a, varsAll, vals2)
b_val = specialize_coeff(out.b, varsAll, vals2)

println("a = ", a_val)
println("b = ", b_val)

In [ ]:
# this agrees with the symmetric system

## Quantum Correction
We now allow for finite $\hbar$ perturbative effects from the tilt
$$N_{\gamma} =  N_{0,\gamma}  \hbar^{2} N_{2,\gamma}  + O(\hbar^3) $$

Following Max's SAGE notebook we use the differentials $\tau_k$  $$\partial_E N_{\gamma} = 2\pi i \int_{T(\gamma)} \tau_{k}$$ 
which follow the recurrence relation:
$$ \tau_{k} = \frac{i p}{H-E} \partial_q \tau_{k-1} + \frac{1}{2} \frac{1}{H-E} \partial_q^2 \tau_{k-2}$$
with initial conditions
$$\tau_0 = \frac{dx\wedge dy}{H-E} \,\,\,\,\,\,\,\,\,\,\, \text{ and }, \,\,\,\,\,\,\,\,\,\,\, \tau_{-1} =0.$$

We construct the operators $g_1 =  \frac{ip}{H-E} \partial_q $
and $g_2 = \frac{1}{2(H-E)} \partial_q^2$ and use the change of 
variables to $x= \sin(q)$ and $y = y/ \cos(q)$ and $z=1$ and go back to the weighted projective 
ring $\mathbb P^{(1,2,1)}$. To use the same field as in the previous parts of the calculation we define 
$\tilde g_1 = -i g_1 = y(z^2-x^2)\delta_1[.]-xyz\delta_2[.]$ and work with the real rationals. 

not need to allow complex coefficients if we wr

In [ ]:
"""
    make_quantum_recurrence_real(f, x, y, z)

Construct the differential operators and polynomial recurrence associated
with the quantum/WKB expansion determined by the defining polynomial `f`.

This function returns a collection of operator closures acting on
polynomials in the same ring as `f`, together with a cached recurrence
for the sequence `c_n`. The recurrence is built from two derived
operators, `g1tilde` and `g2`, and is normalized by
`c_{-1} = 0`, `c_0 = 1`, so that
`c_n = g1tilde(c_{n-1}) - g2(c_{n-2})`.

The conventions used here are:
- `mhat` is the weighted Euler-type operator
- `delta1` and `delta2` are the basic reduced differential operators
- `g1 = i * g1tilde`
- the expansion coefficients satisfy `a_n = i^n c_n`

# Arguments
- `f`: Defining polynomial of the curve/potential.
- `x`, `y`, `z`: Generators of the polynomial ring.

# Returns
A named tuple containing:
- `mhat`: the weighted Euler-style operator
- `delta1`, `delta2`: first-order differential operators derived from `f`
- `g1tilde`, `g2`: operators defining the recurrence
- `c(n)`: memoized recursive access to the coefficient polynomial `c_n`
- `CList(max)`: list `[c(0), ..., c(max)]` computed recursively with cache
- `CListIter(max)`: iterative version producing `[c(0), ..., c(max)]`
- `cache`: internal dictionary storing previously computed `c_n`

# Notes
- The iterative and recursive list constructors should produce the same
  sequence.
- Memoization is used so repeated calls to `c(n)` do not recompute earlier
  coefficients.
- The coefficients live in the same polynomial ring as `f`.
"""
function make_quantum_recurrence(f, x, y, z)
    R = parent(f)
    K = base_ring(R)

    quarter = K(1//4)
    half    = K(1//2)

    mhat(U) = quarter * (2*y*derivative(U, y) +
                         x*derivative(U, x) +
                         z*derivative(U, z) +
                         4*U)

    delta1(U) = z * (derivative(U, x) - derivative(f, x) * mhat(U))
    delta2(U) = derivative(U, y) - derivative(f, y) * mhat(U)

    # g1 = i * g1tilde
    g1tilde(U) = y * ((z^2 - x^2) * delta1(U) -
                      z * x * y * delta2(U))

    g2(U) = (half*z^4 - z^2*x^2 + half*x^4) * delta1(delta1(U)) +
            y*z*(-z^2*x + x^3) * delta2(delta1(U)) +
            half*y^2*x^2 * delta2(z^2 * delta2(U)) +
            half*z*(-z^2 + x^2) * (y*z*delta2(U) + x*delta1(U))

    # c_n with a_n = i^n c_n
    cache = Dict{Int, Any}()
    cache[-1] = zero(R)
    cache[0]  = one(R)

    function c(n::Integer)
        n < -1 && error("c[n] is only defined here for n >= -1")
        haskey(cache, n) && return cache[n]
        cache[n] = g1tilde(c(n - 1)) - g2(c(n - 2))
        return cache[n]
    end

    CList(max::Integer) = [c(n) for n in 0:max]

    function CListIter(max::Integer)
        max < 0 && return Any[]
        out = Any[]
        n  = 0
        cn = one(R)
        bn = zero(R)
        co = zero(R)
        while n <= max
            push!(out, cn)
            n += 1
            bn = co
            co = cn
            cn = g1tilde(cn) - g2(bn)
        end
        return out
    end

    return (
        mhat   = mhat,
        delta1 = delta1,
        delta2 = delta2,
        g1tilde = g1tilde,
        g2     = g2,
        c      = c,
        CList  = CList,
        CListIter = CListIter,
        cache  = cache,
    )
end

In [ ]:
println(typeof(eng.f))
println(typeof(parent(eng.f)))

x, y, z = eng.xs
x, y, z = eng.xs
@assert parent(eng.f) === parent(x) === parent(y) === parent(z)
println("All parents match.")

In [ ]:

x, y, z = eng.xs
ops = make_quantum_recurrence(eng.f, x, y, z)
c0 = ops.c(0)
c1 = ops.c(1)
c2 = ops.c(2)

println("c0 = ", c0)
println("c1 = ", c1)
println("c2 = ", c2)

In [ ]:
# Collect the weighted-homogeneous pieces of a polynomial as (degree => component).
function weighted_homogeneous_components(poly, eng::PFEngine)
    comps = Dict{Int, Any}()
    for comp in values(homogeneous_components(poly))
        iszero(comp) && continue
        d = weighted_degree_homogeneous(comp, eng.weights)
        comps[d] = haskey(comps, d) ? comps[d] + comp : comp
    end
    return sort(collect(comps); by=first)
end

# Inverse of the weighted Euler operator mhat on homogeneous pieces:
# if U has weighted degree d, then
#   mhat(U) = ((d + wdim)/degf) U,
# so mhat^{-1}(U) = U / ((d + wdim)/degf).
function mhat_inv(poly, eng::PFEngine)
    wdim = sum(eng.weights)
    degf = weighted_degree_homogeneous(eng.f, eng.weights)

    out = zero(eng.R)
    for (d, comp) in weighted_homogeneous_components(poly, eng)
        out += comp / ((d + wdim) // degf)
    end
    return out
end

# Principal ideal generated by -∂_E f.
# This is the object called dE in the Sage code.
function dE_ideal(eng::PFEngine)
    return ideal(eng.R, [-eng.dfparams[:E]])
end


# Split poly as
#   poly = q * (-∂_E f) + r
# where r is the remainder mod the principal dE-ideal.
function split_off_dE(poly, eng::PFEngine)
    I = dE_ideal(eng)

    nf = normal_form_with_quotients(
        poly, I;
        lift_fn = (h, J) -> lift_to_generators_weighted(h, J; weights=eng.weights)
    )

    return (
        quotient  = nf.quotients[1],   # since dE-ideal has one generator
        remainder = nf.remainder,
        ideal     = I
    )
end

# Integrate the part of poly that lies in the dE-ideal.
# This is the Julia analogue of
#   IntdE(poly) = mhatInv( lift(poly,dE)[0], J )
#
# It expects poly to lie entirely in the dE-ideal.
function IntdE(poly, eng::PFEngine)
    s = split_off_dE(poly, eng)
    iszero(s.remainder) || error("IntdE expected an element of the dE ideal; nonzero remainder found.")
    return mhat_inv(s.quotient, eng)
end

In [ ]:
# Lift an inhomogeneous target to the generators of J by splitting it into
# weighted-homogeneous components and lifting each component separately.
function lift_to_generators_weighted(target, J; weights)
    R = base_ring(J)
    total = [zero(R) for _ in 1:length(gens(J))]

    for comp in values(homogeneous_components(target))
        iszero(comp) && continue
        q = lift_jacobian_homogeneous(comp, J; weights=weights)
        for i in 1:length(total)
            total[i] += q[i]
        end
    end

    return total
end

# Lift an inhomogeneous target by splitting it into weighted-homogeneous pieces
# and applying the engine's homogeneous lift to each piece separately.
function lift_to_generators_componentwise(target, J; hom_lift)
    R = base_ring(J)
    total = [zero(R) for _ in 1:length(gens(J))]

    # Since the ring is graded, homogeneous_components(target) splits by the
    # weighted grading of the ring.
    for comp in values(homogeneous_components(target))
        iszero(comp) && continue
        q = hom_lift(comp, J)
        for i in 1:length(total)
            total[i] += q[i]
        end
    end

    return total
end

# Convenience wrapper: build the componentwise GD lift associated to an engine.
function make_componentwise_lift(eng::PFEngine)
    hom_lift = eng.lift_fn
    return (h, J) -> lift_to_generators_componentwise(h, J; hom_lift=hom_lift)
end

In [ ]:
gd_lift = make_componentwise_lift(eng)

In [ ]:
# IMPORTANT:
# Mathematica's P[2] = a[2], whereas in the real Julia recurrence a2 = -c2.
P2 = -ops.c(2)   # because old a2 = -c2

gd_lift = make_componentwise_lift(eng)

s = split_off_dE(P2, eng)   # we will patch this below to also use gd_lift

Qint = mhat_inv(s.quotient, eng)

Qint_red = pole_reduce_full(Qint, eng.f, eng.J;
                            lift_fn=gd_lift,
                            weights=eng.weights)

R_red = pole_reduce_full(s.remainder, eng.f, eng.J;
                         lift_fn=gd_lift,
                         weights=eng.weights)

In [ ]:
function split_off_dE(poly, eng::PFEngine)
    I = dE_ideal(eng)
    gd_lift = make_componentwise_lift(eng)

    nf = normal_form_with_quotients(poly, I; lift_fn=gd_lift)

    return (
        quotient  = nf.quotients[1],   # dE-ideal has one generator
        remainder = nf.remainder,
        ideal     = I
    )
end

In [ ]:
# Fast split with respect to the principal ideal (gE), where
#   gE = z^4 - x^2 z^2 = -∂_E f.
#
# We use the identity
#   x^a y^b z^c = x^a y^b z^(c-4) * gE + x^(a+2) y^b z^(c-2),
# recursively until the remaining z-power is < 4.
function split_off_dE_fast(poly, eng::PFEngine)
    x, y, z = eng.xs
    R = eng.R
    gE = -eng.dfparams[:E]

    q = zero(R)
    r = zero(R)

    for i in 1:length(poly)
        e = exponent_vector(poly, i)
        a, b, c = e
        coeff_i = coeff(poly, x^a * y^b * z^c)

        # Reduce this monomial against gE
        qq = zero(R)
        aa, bb, cc = a, b, c

        while cc >= 4
            qq += coeff_i * x^aa * y^bb * z^(cc - 4)
            aa += 2
            cc -= 2
        end

        rr = coeff_i * x^aa * y^bb * z^cc

        q += qq
        r += rr
    end

    return (quotient=q, remainder=r, ideal=ideal(R, [gE]))
end

In [ ]:
P2 = -ops.c(2)   # old a2 = -c2

s = split_off_dE_fast(P2, eng)
gE = -eng.dfparams[:E]

@assert s.quotient * gE + s.remainder == P2
println("fast dE split ok")P2 = -ops.c(2)   # old a2 = -c2


In [ ]:
Qint = mhat_inv(s.quotient, eng)

gd_lift = make_componentwise_lift(eng)

Qint_red = pole_reduce_full(Qint, eng.f, eng.J;
                            lift_fn=gd_lift,
                            weights=eng.weights)

R_red = pole_reduce_full(s.remainder, eng.f, eng.J;
                         lift_fn=gd_lift,
                         weights=eng.weights)

println("Integrated part reduced = ", Qint_red)
println("Remainder reduced       = ", R_red)

In [ ]:
Qcoords = reduce_to_basis(eng, Qint_red)

vars = [eng.params[:mu1], eng.params[:mu2]]
vals = [0, 0]

Q0_00 = specialize_coeff(Qcoords.c0, vars, vals)
Q4_00 = specialize_coeff(Qcoords.c4, vars, vals)

println("On the symmetric slice:")
println("coeff of Ω/f       = ", Q0_00)
println("coeff of z^4Ω/f^2  = ", Q4_00)

In [ ]:
function transform_oldconv_coeff(a, Evar, mu0var)
    num = evaluate(numerator(a),   [Evar, mu0var], [1 - Evar, -mu0var])
    den = evaluate(denominator(a), [Evar, mu0var], [1 - Evar, -mu0var])
    return num // den
end

E   = eng.params[:E]
mu0 = eng.params[:mu0]

Q0_old = transform_oldconv_coeff(Q0_00, E, mu0)
Q4_old = transform_oldconv_coeff(Q4_00, E, mu0)

println("Old-convention coeff of Ω/f      = ", Q0_old)
println("Old-convention coeff of z^4Ω/f^2 = ", Q4_old)

lift into the principal ideal $dE = - \partial_E f = z^4- x^2z^2$

split $\mathrm{poly} = q(-\partial_E f) + r$ where $r$ is the remainder mod the principal ideal 
$\partial_E f$

In [ ]:
# Fast quotient/remainder decomposition with respect to
# gE = -∂_E f = z^4 - x^2 z^2.
#
# This mirrors Sage's S(poly).lift(dE)[0] for the principal ideal dE=(gE).
function split_off_dE_fast(poly, eng::PFEngine)
    x, y, z = eng.xs
    R = eng.R
    gE = -eng.dfparams[:E]

    q = zero(R)
    r = zero(R)

    for i in 1:length(poly)
        e = exponent_vector(poly, i)
        a, b, c = e
        mon = x^a * y^b * z^c
        coeff_i = coeff(poly, mon)

        aa, bb, cc = a, b, c
        qq = zero(R)

        # Use z^4 = x^2 z^2 + gE repeatedly until the remaining z-power is < 4
        while cc >= 4
            qq += coeff_i * x^aa * y^bb * z^(cc - 4)
            aa += 2
            cc -= 2
        end

        rr = coeff_i * x^aa * y^bb * z^cc

        q += qq
        r += rr
    end

    return (quotient=q, remainder=r)
end

# Inverse of the weighted Euler operator on homogeneous pieces.
function mhat_inv(poly, eng::PFEngine)
    wdim = sum(eng.weights)
    degf = weighted_degree_homogeneous(eng.f, eng.weights)

    out = zero(eng.R)
    for comp in values(homogeneous_components(poly))
        iszero(comp) && continue
        d = weighted_degree_homogeneous(comp, eng.weights)
        out += comp / ((d + wdim) // degf)
    end
    return out
end

# Julia analogue of the Sage function IntdE(poly)
function IntdE_fast(poly, eng::PFEngine)
    s = split_off_dE_fast(poly, eng)
    return mhat_inv(s.quotient, eng), s.remainder
end

In [ ]:
P2 = -ops.c(2)   # because old a2 = -c2

Qint, r = IntdE_fast(P2, eng)

println("quotient integrated = ", Qint)
println("remainder           = ", r)

@assert (-eng.dfparams[:E]) * split_off_dE_fast(P2, eng).quotient + r == P2

In [ ]:
gd_lift = make_componentwise_lift(eng)

Qint_red = pole_reduce_full(Qint, eng.f, eng.J;
                            lift_fn=gd_lift,
                            weights=eng.weights)

r_red = pole_reduce_full(r, eng.f, eng.J;
                         lift_fn=gd_lift,
                         weights=eng.weights)

println("Integrated part reduced = ", Qint_red)
println("Remainder reduced       = ", r_red)